# SAM-CapsNet

Capsule Network with Multi-Scale Attention (SAM-CapsNet) for Video-Based Human Activity Recognition

---

| Cell | Purpose |
|---|---|
| 1 | Install |
| 2 | Imports |
| 3 | Dataset (context-window sampling from Paper 2) |
| 4 | Model architecture |
| 5 | Ablation config registry (8 configs) |
| 6 | Training utilities |
| **7** | **⚙️ Config — edit DATASET_PATH here** |
| 8 | Load dataset |
| 9 | Quick single run (verify pipeline) |
| 10 | Full ablation (8 configs × N seeds) |
| **11** | **Per-class accuracy table + heatmap + grouped bar chart** |
| 12 | Load & replot from saved JSON |
| 13 | Attention + CoM visualisation |
| 14 | **Capsule magnitude / activation-attenuation analysis (Sec. II-E)** |
| 15 | **Coupling-coefficient clustering + peak confidence (Fig. 2)** |
| 16 | **Four-part interpretability figure: attention, PrimaryCaps, PCA, polar view (Fig. 3)** |
| 17 | **Inter-class capsule vector analysis (Fig. 4)** |
| 18 | **Confusion-matrix comparison: baseline vs SAM-CapsNet (Fig. 5)** |

## Cell 1 — Install

In [ ]:
%pip install torch torchvision opencv-python matplotlib seaborn scikit-learn tqdm --quiet


## Cell 2 — Imports

In [ ]:
import os, cv2, json, math, random, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, silhouette_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Cell 3 — Dataset

Context-window sampling from Paper 2: for each anchor frame index `i`, collects `i-k` to `i+k` frames and averages them into a single motion-context representation. This gives temporal branch real velocity signal, spatial branch motion-salient inputs, and makes all three branches genuinely complementary — widening the gap between the full model and any single-branch subset.

In [ ]:
import hashlib, pickle

class UCF11VideoDataset(Dataset):
    """
    UCF-11 dataset with fixed-rate sampling at 5fps (from Paper 2).

    Sampling strategy:
      - Fixed-rate anchors: one frame every (native_fps / sample_fps) frames.
        This gives physically comparable motion intervals across clips
        regardless of original recording frame rate.
      - NO context-window averaging: clean single frames are passed to the
        model so attention branches receive sharp, differentiable inputs.

    Why NOT context-window averaging for SAM-CapsNet:
      Context-window averaging works in Paper 2 because the CoM trajectory
      is extracted before averaging -- two separate signals feeding separate
      branches. In SAM-CapsNet, the attention branches (temporal, spatial,
      channel) all need sharp per-frame differences to compute meaningful
      attention maps. Averaging 11 frames homogenises all frame
      representations, removing exactly the signal the attention branches
      need to differentiate themselves. This was confirmed empirically:
      context-window averaging reduced all 8 ablation configs to within
      2.41pp of each other, with the full model scoring LOWEST.

    Disk cache: processed tensors saved on first run, reloaded on
    subsequent epochs/seeds for 10-50x speedup.
    """
    VIDEO_EXTS = ('*.avi', '*.mp4', '*.mov', '*.mpg', '*.mpeg')

    def __init__(self, root_dir, sequence_length=16,
                 target_size=(224, 224), sample_fps=5.0,
                 augment=False, cache_dir='.cache',
                 class_subset=None):
        self.root_dir        = Path(root_dir)
        self.sequence_length = sequence_length
        self.target_size     = target_size
        self.sample_fps      = sample_fps
        self.augment         = augment
        self.cache_dir       = Path(cache_dir) if cache_dir else None
        self.class_subset    = [c.lower() for c in class_subset] if class_subset else None

        aug_tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomRotation(degrees=8),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        clean_tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.transform = aug_tf if augment else clean_tf

        self.samples      = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        self._setup()

        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _setup(self):
        if not self.root_dir.exists():
            raise FileNotFoundError("Dataset not found: {}".format(self.root_dir))
        all_dirs = sorted(d.name for d in self.root_dir.iterdir()
                          if d.is_dir() and not d.name.startswith(('.','__')))
        if self.class_subset is not None:
            classes = [d for d in all_dirs
                       if d.lower() in self.class_subset]
            missing = [c for c in self.class_subset
                       if c not in [x.lower() for x in classes]]
            if missing:
                print('WARNING: these classes not found in dataset: {}'.format(missing))
        else:
            classes = all_dirs
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for i, c in enumerate(classes)}
        print("Found {} classes: {}".format(len(classes), classes))
        for cls in classes:
            p = self.root_dir / cls
            vids = []
            for ext in self.VIDEO_EXTS:
                vids.extend(p.glob(ext))
            for sub in p.iterdir():
                if sub.is_dir() and sub.name != 'annotation':
                    for ext in self.VIDEO_EXTS:
                        vids.extend(sub.glob(ext))
            for vp in vids:
                self.samples.append((str(vp), self.class_to_idx[cls]))
            print("  {}: {} videos".format(cls, len(vids)))
        print("Total: {} videos".format(len(self.samples)))

    def _cache_key(self, path):
        subset_key = ",".join(sorted(self.class_subset)) if self.class_subset else "all"
        sig = "{}|{}|{}|{}|{}".format(
            path, self.sequence_length, self.sample_fps, str(self.target_size), subset_key)
        return hashlib.md5(sig.encode()).hexdigest()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        if self.cache_dir and not self.augment:
            cache_file = self.cache_dir / (self._cache_key(path) + '.pkl')
            if cache_file.exists():
                try:
                    return pickle.loads(cache_file.read_bytes()), label
                except Exception:
                    pass

        frames = self._load_video(path)
        if frames is None:
            frames = torch.zeros(self.sequence_length, 3, *self.target_size)

        if self.cache_dir and not self.augment:
            try:
                cache_file = self.cache_dir / (self._cache_key(path) + '.pkl')
                cache_file.write_bytes(pickle.dumps(frames))
            except Exception:
                pass

        return frames, label

    def _load_video(self, path):
        try:
            cap = cv2.VideoCapture(path)
            if not cap.isOpened():
                return None
            native_fps = cap.get(cv2.CAP_PROP_FPS)
            if native_fps <= 0 or native_fps > 240:
                native_fps = 25.0
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total <= 0:
                cap.release()
                return None

            # Fixed-rate anchors at sample_fps
            step    = max(1, int(round(native_fps / self.sample_fps)))
            anchors = [min(i * step, total - 1) for i in range(self.sequence_length)]

            # Sequential read: one forward pass, no random seeking
            cap.set(cv2.CAP_PROP_POS_FRAMES, anchors[0])
            raw   = {}
            cur   = anchors[0]
            last  = anchors[-1]
            while cur <= last:
                ret, frame = cap.read()
                if ret:
                    raw[cur] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                cur += 1
            cap.release()

            if not raw:
                return None

            frames = []
            for anchor in anchors[:self.sequence_length]:
                # Single clean frame — no averaging
                fi = min(raw.keys(), key=lambda k: abs(k - anchor))
                frames.append(self.transform(raw[fi]))

            while len(frames) < self.sequence_length:
                frames.append(frames[-1])
            return torch.stack(frames[:self.sequence_length])

        except Exception as e:
            print("Error loading {}: {}".format(path, e))
            return None

    def get_labels(self):
        return [s[1] for s in self.samples]

    def clear_cache(self):
        if self.cache_dir and self.cache_dir.exists():
            import shutil
            shutil.rmtree(self.cache_dir)
            self.cache_dir.mkdir()
            print("Cache cleared.")


def make_stratified_splits(dataset, train_r=0.70, val_r=0.15, seed=42):
    labels = np.array(dataset.get_labels())
    n = len(labels)
    sss1 = StratifiedShuffleSplit(1, test_size=1-train_r-val_r, random_state=seed)
    tv_idx, test_idx = next(sss1.split(np.zeros(n), labels))
    val_frac = val_r / (train_r + val_r)
    sss2 = StratifiedShuffleSplit(1, test_size=val_frac, random_state=seed)
    rel_tr, rel_val = next(sss2.split(np.zeros(len(tv_idx)), labels[tv_idx]))
    return tv_idx[rel_tr].tolist(), tv_idx[rel_val].tolist(), test_idx.tolist()


print("Dataset class defined (5fps fixed-rate, clean frames, disk cache).")


## Cell 4 — Model Architecture


In [ ]:
class MultiScaleSAMAttention(nn.Module):
    """
    Sequential multiplicative attention: Channel -> Spatial (temporal-guided)
    -> Temporal residual.

    Channel and spatial use standard multiplicative CBAM-style attention
    (contrast enhancement, not uniform suppression).
    Temporal uses signed residual (tanh gate) to allow feature amplification.
    """
    def __init__(self, in_channels=256, reduction_ratio=8,
                 use_channel=True, use_spatial=True, use_point_prompt=True,
                 use_temporal=True, use_learnable_fusion=True):
        super().__init__()
        self.use_channel          = use_channel
        self.use_spatial          = use_spatial
        self.use_point_prompt     = use_point_prompt
        self.use_temporal         = use_temporal
        self.use_learnable_fusion = use_learnable_fusion
        r = max(1, in_channels // reduction_ratio)

        # ── Channel: CBAM-style squeeze-excitation ────────────────────────
        if use_channel:
            self.ch_gap  = nn.AdaptiveAvgPool2d(1)
            self.ch_gmp  = nn.AdaptiveMaxPool2d(1)   # max pool adds selectivity
            self.ch_fc1  = nn.Linear(in_channels, r, bias=False)
            self.ch_fc2  = nn.Linear(r, in_channels, bias=False)

        # ── Point prompt ──────────────────────────────────────────────────
        if use_point_prompt:
            self.point_prompt = nn.Parameter(torch.zeros(1, in_channels, 1, 1))
            nn.init.normal_(self.point_prompt, std=0.1)  # paper Sec. II.F: sigma = 0.1

        # ── Temporal: dual-scale motion map ───────────────────────────────
        if use_temporal:
            self.tm_fine   = nn.Sequential(
                nn.Conv2d(in_channels*2, r//2, 3, padding=1, bias=False),
                nn.BatchNorm2d(r//2), nn.ReLU(True))
            self.tm_coarse = nn.Sequential(
                nn.Conv2d(in_channels*2, r//2, 5, padding=2, bias=False),
                nn.BatchNorm2d(r//2), nn.ReLU(True))
            self.tm_fuse   = nn.Sequential(
                nn.Conv2d(r, 1, 1, bias=False), nn.Sigmoid())
            self.tm_drop   = nn.Dropout2d(p=0.1)
            # Signed gate: tanh allows amplification of motion features
            self.alpha_tm  = nn.Parameter(torch.tensor(0.1))

        # ── Spatial: guided by temporal map ───────────────────────────────
        if use_spatial:
            extra = 1 if use_temporal else 0
            self.sp_guide = nn.Conv2d(in_channels + extra, in_channels,
                                       1, bias=False)
            self.sp_conv  = nn.Sequential(
                nn.Conv2d(in_channels, r, 1, bias=False),
                nn.BatchNorm2d(r), nn.ReLU(True),
                nn.Conv2d(r, r, 3, padding=1, bias=False),
                nn.BatchNorm2d(r), nn.ReLU(True),
                nn.Conv2d(r, 1, 1, bias=False), nn.Sigmoid())
            self.sp_drop  = nn.Dropout2d(p=0.1)

        if use_learnable_fusion:
            self.lam = nn.Parameter(torch.tensor(0.5))
        else:
            self.register_buffer('lam', torch.tensor(0.5))

    def forward(self, x, prev_x=None, return_maps=False):
        B, C, H, W = x.shape
        out = x
        a_m, a_s, ac = None, None, None

        # Step 1: Channel attention (multiplicative, CBAM-style)
        if self.use_channel:
            avg = self.ch_gap(x).view(B, C)
            mx  = self.ch_gmp(x).view(B, C)
            # Shared FC on both poolings, summed
            ac  = torch.sigmoid(
                self.ch_fc2(F.relu(self.ch_fc1(avg))) +
                self.ch_fc2(F.relu(self.ch_fc1(mx)))
            ).view(B, C, 1, 1)
            out = out * ac   # multiplicative: contrast enhancement

        # Step 2: Temporal motion map
        if self.use_temporal and prev_x is not None:
            cat = torch.cat([x, prev_x], dim=1)
            a_m = self.tm_fuse(
                torch.cat([self.tm_fine(cat), self.tm_coarse(cat)], dim=1))
            if self.training:
                a_m = self.tm_drop(a_m)

        # Step 3: Spatial attention (applied to channel-refined features)
        if self.use_spatial:
            inp = out + self.point_prompt if self.use_point_prompt else out
            if a_m is not None:
                inp = self.sp_guide(torch.cat([inp, a_m], dim=1))
            a_s = self.sp_conv(inp)
            if self.training:
                a_s = self.sp_drop(a_s)
            out = out * a_s   # multiplicative: spatial focus

        # Step 4: Temporal residual (signed gate allows amplification)
        if a_m is not None:
            alpha = torch.tanh(self.alpha_tm)   # in (-1, 1)
            out   = out + alpha * (out * a_m)   # amplify motion regions

        # Build visualisation maps (reuse cached tensors)
        if return_maps:
            lam = self.lam.clamp(0.0, 1.0) if hasattr(self, 'lam') else torch.tensor(0.5)
            if a_m is not None and a_s is not None:
                a_f = lam * a_s + (1.0 - lam) * a_m
            elif a_s is not None:
                a_f = a_s
            elif a_m is not None:
                a_f = a_m
            else:
                a_f = torch.ones(B, 1, H, W, device=x.device)
            if ac is None:
                ac = torch.ones(B, C, 1, 1, device=x.device)
            return out, a_f, ac

        return out, None, None

    def gate_values(self):
        """For logging/monitoring during training."""
        gv = {}
        if self.use_temporal and hasattr(self, 'alpha_tm'):
            gv['alpha_tm'] = torch.tanh(self.alpha_tm)
        return gv


class PrimaryCaps(nn.Module):
    def __init__(self, in_channels=256, out_capsules=32, out_dim=8):
        super().__init__()
        self.out_capsules, self.out_dim = out_capsules, out_dim
        self.caps_conv = nn.Conv2d(in_channels, out_capsules * out_dim,
                                   kernel_size=3, padding=1)

    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)

    def forward(self, x):
        B = x.size(0)
        x = self.caps_conv(x)
        H, W = x.shape[2], x.shape[3]
        x = x.view(B, self.out_capsules, self.out_dim, H, W)
        x = x.permute(0, 3, 4, 1, 2).contiguous().view(B, -1, self.out_dim)
        return self.squash(x)


class ActivityCaps(nn.Module):
    def __init__(self, in_capsules, in_dim=8, out_capsules=11, out_dim=16,
                 routing_iters=3):
        super().__init__()
        self.out_capsules  = out_capsules
        self.out_dim       = out_dim
        self.routing_iters = routing_iters
        self.W = nn.Parameter(
            0.01 * torch.randn(1, in_capsules, out_capsules, out_dim, in_dim))

    def squash(self, v, eps=1e-8):
        n = v.norm(dim=-1, keepdim=True)
        return (n**2 / (1 + n**2)) * v / (n + eps)

    def forward(self, u, return_routing=False):
        B = u.shape[0]
        W = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        b = torch.zeros(B, u.shape[1], self.out_capsules, 1, device=u.device)
        c = None
        for r in range(self.routing_iters):
            c = F.softmax(b, dim=2)
            s = (c * u_hat).sum(dim=1)
            v = self.squash(s)
            if r < self.routing_iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(dim=-1, keepdim=True)
        if return_routing:
            # u_hat: (B, N_primary, num_classes, out_dim) prediction vectors
            # c:     (B, N_primary, num_classes, 1) final coupling coefficients
            # s:     (B, num_classes, out_dim) pre-squash weighted sum (routing logit input)
            return v, dict(u_hat=u_hat, c=c, s=s)
        return v


class SAMCapsNet(nn.Module):
    # NOTE (edit): FEAT_H/FEAT_W/PRIM_CAPS used to be hardcoded class constants
    # (7, 7, 16) valid only for 112x112 input -- the memory-reduced config this
    # notebook builds for the 101-class run. This cell's own comment implies a
    # different original config for the 11-class experiments ("16 caps at 7x7
    # -> 784 in_caps (vs 6272 for 224x224 with 32 caps)" -- 32*14*14 = 6272).
    # Made input_size/prim_caps constructor args, with feature-map size computed
    # from an actual forward pass through the backbone, so both configs (and any
    # other input resolution) work correctly instead of only 112x112/16 caps.
    PRIM_DIM = 8

    def __init__(self, num_classes=11, sam_config=None, input_size=(112, 112),
                 prim_caps=16):
        super().__init__()
        self.num_classes = num_classes
        self.use_sam     = (sam_config is not None)
        self.PRIM_CAPS   = prim_caps

        self.conv = nn.Sequential(
            nn.Conv2d(3,   64,  7, stride=2, padding=3),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64,  128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
        )

        if self.use_sam:
            cfg = dict(use_channel=True, use_spatial=True,
                       use_point_prompt=True, use_temporal=True,
                       use_learnable_fusion=True)
            cfg.update(sam_config)
            self.attention = MultiScaleSAMAttention(in_channels=256, **cfg)

        with torch.no_grad():
            _feat = self.conv(torch.zeros(1, 3, *input_size))
        self.FEAT_H, self.FEAT_W = _feat.shape[2], _feat.shape[3]

        in_caps = self.PRIM_CAPS * self.FEAT_H * self.FEAT_W
        self.pre_caps_norm = nn.LayerNorm([256, self.FEAT_H, self.FEAT_W])
        self.primary  = PrimaryCaps(256, self.PRIM_CAPS, self.PRIM_DIM)
        self.activity = ActivityCaps(in_caps, self.PRIM_DIM, num_classes, 16)

    def forward(self, x, return_attention=False, return_routing=False):
        B, T, C, H, W = x.shape

        # Batch all frames through conv in one call
        x_flat     = x.view(B * T, C, H, W)
        feats_flat = self.conv(x_flat)
        feats      = feats_flat.view(B, T, 256, self.FEAT_H, self.FEAT_W)

        attended, sp_maps, ch_maps = [], [], []
        for t in range(T):
            if self.use_sam:
                prev = feats[:, t-1] if t > 0 else None
                f, sa, ca = self.attention(
                    feats[:, t], prev, return_maps=return_attention)
                attended.append(f)
                if return_attention and sa is not None:
                    sp_maps.append(sa)
                    ch_maps.append(ca)
            else:
                attended.append(feats[:, t])

        agg  = torch.stack(attended, dim=1).mean(dim=1)
        agg  = self.pre_caps_norm(agg)
        caps = self.primary(agg)   # PrimaryCaps output (B, N_primary, PRIM_DIM)

        if return_routing:
            out, routing_info = self.activity(caps, return_routing=True)
        else:
            out = self.activity(caps)

        # NOTE (added): return_routing exposes the intermediate capsule tensors
        # needed for the paper's interpretability analyses (Sec. II-E activation
        # attenuation, Fig. 2 coupling-coefficient clustering, Fig. 3b/c/d,
        # Fig. 4 per-class capsule-vector distributions) that the original
        # notebook had no code path to produce. return_attention is unchanged
        # and remains backward compatible with existing call sites.
        extras = {}
        if return_attention and self.use_sam and sp_maps:
            extras['spatial']      = torch.stack(sp_maps, dim=1)
            extras['channel']      = torch.stack(ch_maps, dim=1)
            extras['feature_size'] = (self.FEAT_H, self.FEAT_W)
            extras['input_size']   = (H, W)
        if return_routing:
            extras['primary'] = caps                  # (B, N_primary, PRIM_DIM)
            extras['u_hat']   = routing_info['u_hat']  # (B, N_primary, num_classes, 16)
            extras['c']       = routing_info['c']      # (B, N_primary, num_classes, 1)
            extras['s']       = routing_info['s']      # (B, num_classes, 16)

        if extras:
            return out, extras
        return out

    def gate_sparsity_loss(self, lam=0.0):
        # No sparsity loss needed with multiplicative attention
        # (branches can be zeroed by the attention maps themselves)
        return torch.tensor(0.0, requires_grad=True, device=
                            next(self.parameters()).device)


print("SAMCapsNet with CBAM-style sequential multiplicative attention defined.")

_d = torch.zeros(2, 16, 3, 112, 112)
_m = SAMCapsNet(num_classes=11, sam_config={})
with torch.no_grad():
    _o = _m(_d)
assert tuple(_o.shape) == (2, 11, 16)
print("Forward pass OK:", tuple(_o.shape))

# Verify no suppression-only behaviour
print()
print("Channel attention sample output (should have values near 1.0 for")
print("important channels, not uniformly small):")
_f = torch.randn(2, 256, 14, 14)
_att = _m.attention
with torch.no_grad():
    avg = _att.ch_gap(_f).view(2, 256)
    mx  = _att.ch_gmp(_f).view(2, 256)
    ac  = torch.sigmoid(_att.ch_fc2(F.relu(_att.ch_fc1(avg))) +
                        _att.ch_fc2(F.relu(_att.ch_fc1(mx))))
print("  ac min={:.3f}  max={:.3f}  mean={:.3f}".format(
      ac.min().item(), ac.max().item(), ac.mean().item()))
print("  (min near 0, max near 1 = contrast enhancement working)")

del _d, _m, _o, _f, _att, ac

# Smoke-test the new return_routing path (added for the interpretability analyses
# in Cells 14-19 below): checks shapes only, not correctness of trained behaviour.
_d2 = torch.zeros(2, 16, 3, 112, 112)
_m2 = SAMCapsNet(num_classes=11, sam_config={})
with torch.no_grad():
    _out2, _extra2 = _m2(_d2, return_attention=True, return_routing=True)
assert tuple(_out2.shape) == (2, 11, 16)
assert tuple(_extra2['primary'].shape[0::2]) == (2, 8)          # (B, ..., PRIM_DIM)
assert _extra2['u_hat'].shape[0] == 2 and _extra2['u_hat'].shape[2] == 11
assert _extra2['c'].shape[0] == 2 and _extra2['c'].shape[2] == 11
assert 'spatial' in _extra2 and 'channel' in _extra2
print("return_routing smoke test OK -- primary:", tuple(_extra2['primary'].shape),
      " u_hat:", tuple(_extra2['u_hat'].shape), " c:", tuple(_extra2['c'].shape))
del _d2, _m2, _out2, _extra2


## Cell 4B — Non-Capsule Baseline Models





In [ ]:
class BaselineCNN(nn.Module):
    """Table 1, row 1: 'Baseline CNN (softmax)'.
    Identical conv backbone to SAMCapsNet, no attention, no capsule routing:
    temporal-mean-pooled features -> linear softmax classifier."""
    def __init__(self, num_classes=11):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,   64,  7, stride=2, padding=3),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64,  128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(256, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        feats = self.conv(x.view(B * T, C, H, W))
        feats = self.gap(feats).view(B, T, 256)
        agg   = feats.mean(dim=1)          # temporal mean pooling
        return self.fc(agg)                # raw logits, (B, num_classes)


class CNNSAMNoCapsule(nn.Module):
    """Table 1, row 2: 'CNN + Multi-Scale SAM'.
    Same backbone + full MultiScaleSAMAttention as SAM-CapsNet, but PrimaryCaps /
    ActivityCaps is replaced with a plain FC softmax classifier (no routing)."""
    def __init__(self, num_classes=11):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,   64,  7, stride=2, padding=3),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64,  128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
        )
        self.attention = MultiScaleSAMAttention(
            in_channels=256, use_channel=True, use_spatial=True,
            use_point_prompt=True, use_temporal=True, use_learnable_fusion=True)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(256, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        feats = self.conv(x.view(B * T, C, H, W)).view(B, T, 256, 7, 7)
        attended = []
        for t in range(T):
            prev = feats[:, t - 1] if t > 0 else None
            f, _, _ = self.attention(feats[:, t], prev, return_maps=False)
            attended.append(f)
        agg = torch.stack(attended, dim=1).mean(dim=1)
        agg = self.gap(agg).view(B, 256)
        return self.fc(agg)                # raw logits, (B, num_classes)


print("BaselineCNN and CNNSAMNoCapsule defined.")
_d = torch.zeros(2, 16, 3, 112, 112)
for _cls in (BaselineCNN, CNNSAMNoCapsule):
    _m = _cls(num_classes=11)
    with torch.no_grad():
        _o = _m(_d)
    assert tuple(_o.shape) == (2, 11), _cls.__name__
    print("  {:<16} forward OK: {}".format(_cls.__name__, tuple(_o.shape)))
del _d, _m, _o


## Cell 5 — Ablation Configurations (8 configs)

In order: Proposed → Pure CapsNet → Channel → Temporal → Spatial → Channel+Temporal → Temporal+Spatial → Channel+Spatial

In [ ]:
# Ablation configurations — 8 entries in the order specified
ABLATION_CONFIGS = [
    ('full_sam',
     dict(use_channel=True, use_spatial=True, use_point_prompt=True,
          use_temporal=True, use_learnable_fusion=True),
     'SAM-CapsNet (Proposed)',
     'Full model: channel + spatial + point prompt + temporal + learnable fusion'),

    ('no_sam',
     None,
     'Pure CapsNet',
     'Capsule network only, no attention'),

    ('ch_only',
     dict(use_channel=True,  use_spatial=False, use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Channel only',
     'Channel attention branch only'),

    ('tm_only',
     dict(use_channel=False, use_spatial=False, use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Temporal only',
     'Temporal motion attention branch only'),

    ('sp_only',
     dict(use_channel=False, use_spatial=True,  use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Spatial only',
     'Spatial attention branch only (no point prompt)'),

    ('ch_tm',
     dict(use_channel=True,  use_spatial=False, use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Channel + Temporal',
     'Channel and temporal branches'),

    ('tm_sp',
     dict(use_channel=False, use_spatial=True,  use_point_prompt=False,
          use_temporal=True,  use_learnable_fusion=False),
     'Temporal + Spatial',
     'Temporal and spatial branches (no channel, no PP)'),

    ('ch_sp',
     dict(use_channel=True,  use_spatial=True,  use_point_prompt=False,
          use_temporal=False, use_learnable_fusion=False),
     'Channel + Spatial',
     'Channel and spatial branches (no temporal, no PP)'),
]

CONFIG_BY_KEY = {k: (k, cfg, lbl, desc) for k, cfg, lbl, desc in ABLATION_CONFIGS}
print("{} ablation configurations registered.".format(len(ABLATION_CONFIGS)))
for k, _, lbl, _ in ABLATION_CONFIGS:
    print("  {:<28} {}".format(k, lbl))


## Cell 6 — Training Utilities

In [ ]:
SEEDS = [42, 123, 456, 789, 2024]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(model, loader, optimizer, device, grad_clip=5.0, grad_accum=4):
    """
    Gradient accumulation + mixed precision for memory efficiency.
    grad_accum=4 with batch_size=2 -> effective batch=8 without the RAM cost.
    Mixed precision (autocast) halves activation memory on CUDA GPUs.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    optimizer.zero_grad()

    for step, (x, y) in enumerate(tqdm(loader, desc="  train", leave=False)):
        x, y = x.to(device), y.to(device)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(x).norm(dim=-1)
                loss   = criterion(logits, y) / grad_accum
            scaler.scale(loss).backward()
        else:
            logits = model(x).norm(dim=-1)
            loss   = criterion(logits, y) / grad_accum
            loss.backward()

        total_loss += loss.item() * grad_accum
        correct    += (logits.detach().argmax(1) == y).sum().item()
        n          += y.size(0)

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            if scaler is not None:
                if grad_clip:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                if grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            optimizer.zero_grad()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    return total_loss / len(loader), correct / n * 100


def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="  eval", leave=False):
        p = model(x.to(device)).norm(dim=-1).argmax(1).cpu()
        preds.extend(p.numpy())
        labels.extend(y.numpy())
    return (np.array(preds) == np.array(labels)).mean() * 100, np.array(preds), np.array(labels)


def run_one_seed(dataset, num_classes, sam_config, seed, num_epochs,
                 batch_size, lr, device, input_size=(112, 112), prim_caps=16):
    set_seed(seed)
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=seed)

    # FIX 6: num_workers=2 for parallel data loading
    nw = 1 if os.name != 'nt' else 0   # keep low for UCF-101 memory
    train_ld = DataLoader(Subset(dataset, train_idx), batch_size=batch_size,
                          shuffle=True,  num_workers=nw, pin_memory=True)
    val_ld   = DataLoader(Subset(dataset, val_idx),   batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)
    test_ld  = DataLoader(Subset(dataset, test_idx),  batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)

    model = SAMCapsNet(num_classes=num_classes, sam_config=sam_config,
                       input_size=input_size, prim_caps=prim_caps).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Paper Sec. II.F: "Optimisation used Adam ... with step decay (gamma = 0.5 every 7 epochs)"
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=7, gamma=0.5)
    best_val, best_state, train_accs, val_accs = 0.0, None, [], []

    for ep in range(num_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_ld, opt, device,
                                          grad_accum=GRAD_ACCUM)
        val_acc, _, _   = evaluate(model, val_ld, device)
        scheduler.step()
        train_accs.append(tr_acc)
        val_accs.append(val_acc)
        if val_acc > best_val:
            best_val   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        gate_str = ""
        if model.use_sam and hasattr(model.attention, 'gate_values'):
            gv = model.attention.gate_values()
            if gv:
                gate_str = "  [" + " ".join(
                    "{}={:.3f}".format(k, v.item()) for k, v in gv.items()) + "]"
        print("    Ep {:02d}/{} | loss {:.4f} | train {:.1f}% | val {:.1f}%{}".format(
              ep + 1, num_epochs, tr_loss, tr_acc, val_acc, gate_str))

    model.load_state_dict(best_state)
    test_acc, preds, gt = evaluate(model, test_ld, device)
    return dict(test_acc=test_acc, best_val=best_val,
                train_accs=train_accs, val_accs=val_accs,
                preds=preds, gt=gt, model=model)


def plot_training_curves(train_accs, val_accs, title="", save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ep = range(1, len(train_accs) + 1)
    ax.plot(ep, train_accs, "b-o", ms=3, label="Train")
    ax.plot(ep, val_accs,   "r-o", ms=3, label="Val")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_title("Training curves" + ("  -  " + title if title else ""))
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()


def plot_confusion_matrix(gt, preds, class_names, title="", save_path=None):
    cm_arr = confusion_matrix(gt, preds)
    acc    = (np.array(preds) == np.array(gt)).mean() * 100
    sz     = max(8, len(class_names))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    sns.heatmap(cm_arr, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title("Confusion Matrix" + ("  -  " + title if title else "")
                 + "\nAcc = {:.2f}%".format(acc))
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()


def plot_ablation_bar(results, save_path=None):
    labels = [r["label"]     for r in results.values()]
    means  = [r["mean_test"] for r in results.values()]
    stds   = [r["std_test"]  for r in results.values()]
    colors = ["#FF5722" if k == "full_sam" else "#2196F3" for k in results]
    fig, ax = plt.subplots(figsize=(11, max(5, len(labels) * 0.65)))
    y = np.arange(len(labels))
    bars = ax.barh(y, means, xerr=stds, color=colors, capsize=4,
                   height=0.6, edgecolor="white", linewidth=0.5)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel("Mean Test Accuracy (%)", fontsize=12)
    ax.set_title("SAM-CapsNet Component Ablation Study (UCF-11)",
                 fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.3); ax.set_xlim(0, max(means) + 14)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_width() + s + 0.5,
                bar.get_y() + bar.get_height() / 2,
                "{:.2f} +/- {:.2f}%".format(m, s), va="center", fontsize=9.5)
    ax.legend(handles=[mpatches.Patch(color="#2196F3", label="Ablation variant"),
                       mpatches.Patch(color="#FF5722", label="SAM-CapsNet (Proposed)")],
              loc="lower right", fontsize=10)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def plot_component_heatmap(results, save_path=None):
    comp_cols  = ["Channel", "Spatial", "Point\nPrompt", "Temporal", "Learnable\nlam"]
    comp_flags = {
        "full_sam": [1,1,1,1,1], "no_sam":  [0,0,0,0,0],
        "ch_only":  [1,0,0,0,0], "tm_only": [0,0,0,1,0],
        "sp_only":  [0,1,0,0,0], "ch_tm":   [1,0,0,1,0],
        "tm_sp":    [0,1,0,1,0], "ch_sp":   [1,1,0,0,0],
    }
    keys   = [k for k in results if k in comp_flags]
    labels = [results[k]["label"] for k in keys]
    matrix = np.array([comp_flags[k] for k in keys], dtype=float)
    accs   = [results[k]["mean_test"] for k in keys]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(5, len(keys) * 0.7)),
                                    gridspec_kw={"width_ratios": [3, 1]})
    annot = [["Y" if v else "" for v in row] for row in matrix]
    sns.heatmap(matrix, ax=ax1, cmap="Blues", vmin=0, vmax=1,
                xticklabels=comp_cols, yticklabels=labels,
                linewidths=0.5, linecolor="white", cbar=False,
                annot=annot, fmt="", annot_kws={"size": 13})
    ax1.set_title("Component Presence", fontweight="bold"); ax1.tick_params(labelsize=9)
    clrs = ["#FF5722" if k == "full_sam" else "#2196F3" for k in keys]
    ax2.barh(np.arange(len(keys)), accs, color=clrs, height=0.6)
    ax2.set_yticks([]); ax2.set_xlabel("Test Acc (%)")
    ax2.set_title("Accuracy", fontweight="bold")
    ax2.set_xlim(0, max(accs) + 14); ax2.grid(axis="x", alpha=0.3)
    for i, a in enumerate(accs):
        ax2.text(a + 0.5, i, "{:.1f}%".format(a), va="center", fontsize=9.5)
    plt.suptitle("SAM-CapsNet: Component presence vs accuracy", fontweight="bold")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


print("Training utilities defined (num_workers=2, pin_memory=True).")


## Cell 6B — Training Loop for Non-Capsule Models



In [ ]:
def train_one_epoch_logits(model, loader, optimizer, device, grad_clip=5.0, grad_accum=4):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    optimizer.zero_grad()

    for step, (x, y) in enumerate(tqdm(loader, desc="  train", leave=False)):
        x, y = x.to(device), y.to(device)
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss   = criterion(logits, y) / grad_accum
            scaler.scale(loss).backward()
        else:
            logits = model(x)
            loss   = criterion(logits, y) / grad_accum
            loss.backward()

        total_loss += loss.item() * grad_accum
        correct    += (logits.detach().argmax(1) == y).sum().item()
        n          += y.size(0)

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            if scaler is not None:
                if grad_clip:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                if grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            optimizer.zero_grad()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    return total_loss / len(loader), correct / n * 100


def evaluate_logits(model, loader, device):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="  eval", leave=False):
        p = model(x.to(device)).argmax(1).cpu()
        preds.extend(p.numpy())
        labels.extend(y.numpy())
    return (np.array(preds) == np.array(labels)).mean() * 100, np.array(preds), np.array(labels)


def run_one_seed_logits(model_ctor, dataset, num_classes, seed, num_epochs,
                        batch_size, lr, device, grad_accum=None):
    """Same protocol as run_one_seed (Cell 6), but for models that output raw
    logits (BaselineCNN, CNNSAMNoCapsule, ResNet18Baseline, MobileNetV2Baseline)."""
    if grad_accum is None:
        grad_accum = GRAD_ACCUM
    set_seed(seed)
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=seed)

    nw = 1 if os.name != 'nt' else 0
    train_ld = DataLoader(Subset(dataset, train_idx), batch_size=batch_size,
                          shuffle=True,  num_workers=nw, pin_memory=True)
    val_ld   = DataLoader(Subset(dataset, val_idx),   batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)
    test_ld  = DataLoader(Subset(dataset, test_idx),  batch_size=batch_size,
                          shuffle=False, num_workers=nw, pin_memory=True)

    model = model_ctor(num_classes).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=7, gamma=0.5)

    best_val, best_state, train_accs, val_accs = 0.0, None, [], []
    for ep in range(num_epochs):
        tr_loss, tr_acc = train_one_epoch_logits(model, train_ld, opt, device, grad_accum=grad_accum)
        val_acc, _, _   = evaluate_logits(model, val_ld, device)
        scheduler.step()
        train_accs.append(tr_acc)
        val_accs.append(val_acc)
        if val_acc > best_val:
            best_val   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print("    Ep {:02d}/{} | loss {:.4f} | train {:.1f}% | val {:.1f}%".format(
              ep + 1, num_epochs, tr_loss, tr_acc, val_acc))

    model.load_state_dict(best_state)
    test_acc, preds, gt = evaluate_logits(model, test_ld, device)
    return dict(test_acc=test_acc, best_val=best_val,
                train_accs=train_accs, val_accs=val_accs,
                preds=preds, gt=gt, model=model)


print("Logits training loop defined (train_one_epoch_logits, evaluate_logits, run_one_seed_logits).")


## Cell 7A — Configuration for Tables 1 & 2 (11-class primary experiment)



In [ ]:
# ── EDIT THIS: point at your local UCF-101 root ─────────────────────────────
DATASET_PATH = '/projects/ucf-101'   # path to UCF-101 root (absolute path)
SAVE_DIR     = 'ablation_results_11class'   # Tables 1 & 2 results go here

NUM_EPOCHS    = 15      # paper Sec. II.F: fixed 15 epochs, no early stopping
BATCH_SIZE    = 2
GRAD_ACCUM    = 4       # effective batch = 2 * 4 = 8
LR            = 5e-4
NUM_SEEDS     = 5

SAMPLE_FPS    = 5.0
CACHE_DIR     = '.cache_ucf101_11class'

INPUT_SIZE    = (224, 224)
PRIM_CAPS     = 32      # matches this notebook's own comment in Cell 4: "vs 6272
                         # for 224x224 with 32 caps" -- the config implied for the
                         # 11-class experiments, distinct from the 101-class run's
                         # memory-reduced 16-cap/112x112 setup (Cell 7B)
SEQ_LEN       = 16      # paper Sec. II.B: T = 16, selected via grid search (Cell 7D)

# 11-class subset used for Tables 1-2 (paper Table 2 row labels -> UCF-101 folder names)
CLASS_SUBSET = [
    'Basketball', 'Biking', 'Diving', 'GolfSwing', 'HorseRiding',
    'SoccerJuggling', 'Swing', 'TennisSwing', 'TrampolineJumping',
    'VolleyballSpiking', 'WalkingWithDog',   # paper's "Walking" = UCF-101's WalkingWithDog
]
assert len(CLASS_SUBSET) == 11

SEEDS = [42, 123, 456, 789, 2024][:NUM_SEEDS]
os.makedirs(SAVE_DIR, exist_ok=True)

print("Dataset     :", DATASET_PATH)
print("Classes     :", len(CLASS_SUBSET), CLASS_SUBSET)
print("Input size  :", INPUT_SIZE)
print("Epochs      :", NUM_EPOCHS, "  Batch:", BATCH_SIZE, "x Accum:", GRAD_ACCUM,
      " = effective", BATCH_SIZE * GRAD_ACCUM)
print("Seeds       :", SEEDS)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


## Cell 7B — Configuration for Table 3's full-101-class row  ← **Edit DATASET_PATH here**



In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
DATASET_PATH = '/projects/ucf-101'  # path to UCF-101 root (absolute path)
SAVE_DIR     = 'ablation_results_ucf101_full'   # separate dir: do NOT overwrite the
                                                 # existing 13-class ablation_results/
                                                 # (that run backs Tables 1-2 in the paper)

# Memory-safe settings for UCF-101 on 8GB GPU
NUM_EPOCHS    = 15      # enough for convergence; cosine LR compresses well
BATCH_SIZE    = 2       # small batch; gradient accumulation makes up for it
GRAD_ACCUM    = 4       # effective batch = 2 * 4 = 8 (same as original B=4)
LR            = 5e-4
NUM_SEEDS     = 5

SAMPLE_FPS    = 5.0
CACHE_DIR     = '.cache_ucf101_full'   # separate cache: full 101-class videos are a
                                        # different (much larger) sample set than the
                                        # 13-class subset, and the cache key already
                                        # differs by class_subset -- separate dirs just
                                        # keep them from cluttering the same folder

# Input size: 112x112 instead of 224x224
#   -> feature map 7x7 instead of 14x14
#   -> in_caps = 16 * 7 * 7 = 784  (vs 6272 for 224x224 with 32 caps)
#   -> ActivityCaps W matrix scales with num_classes; 112x112 keeps this
#      fitting in 8GB even at 101 classes
INPUT_SIZE    = (112, 112)
PRIM_CAPS     = 16      # memory-reduced config for the 101-class run (see Cell 4)
SEQ_LEN       = 16

# Full UCF-101: all 101 classes (CLASS_SUBSET=None discovers every class
# folder under DATASET_PATH automatically -- see Cell 8 / NUM_CLASSES).
# This is ~13,320 clips total vs. 938 for the 13-class subset used in
# Tables 1-2, so expect roughly an order of magnitude more per-epoch time,
# on top of whatever the first-run disk-cache build costs.
CLASS_SUBSET = None

# ─────────────────────────────────────────────────────────────────────────────
SEEDS = [42, 123, 456, 789, 2024][:NUM_SEEDS]
os.makedirs(SAVE_DIR, exist_ok=True)

print("Dataset     :", DATASET_PATH)
print("Classes     :", len(CLASS_SUBSET) if CLASS_SUBSET else "all")
print("Input size  :", INPUT_SIZE)
print("Epochs      :", NUM_EPOCHS, "  Batch:", BATCH_SIZE, "x Accum:", GRAD_ACCUM,
      " = effective", BATCH_SIZE * GRAD_ACCUM)
print("Seeds       :", SEEDS)

# Set env var to reduce memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


## Cell 8 — Load Dataset

## Cell 8A — Dataset Path Diagnostic

In [ ]:
import os
from pathlib import Path

print("Notebook's current working directory:")
print(" ", os.getcwd())
print()

p = Path(DATASET_PATH)
print("DATASET_PATH as configured:", repr(DATASET_PATH))
print("Resolves to (absolute):    ", p.resolve())
print("Exists:                    ", p.exists())
print()

# Walk up from cwd looking for anything named like a UCF-101/UCF-11 folder,
# so you can see what the real path should be.
print("Looking for likely dataset folders under the current directory (depth<=3)...")
found = []
cwd = Path.cwd()
for depth1 in cwd.iterdir():
    if depth1.is_dir() and not depth1.name.startswith('.'):
        if 'ucf' in depth1.name.lower():
            found.append(depth1)
        try:
            for depth2 in depth1.iterdir():
                if depth2.is_dir() and 'ucf' in depth2.name.lower():
                    found.append(depth2)
        except PermissionError:
            pass

if found:
    print("Candidates found:")
    for f in found:
        n_subdirs = sum(1 for x in f.iterdir() if x.is_dir()) if f.is_dir() else 0
        print("  -", f.resolve(), " ({} subfolders)".format(n_subdirs))
else:
    print("  No folder with 'ucf' in the name found within 2 levels of cwd.")
    print("  Contents of cwd:")
    for x in sorted(cwd.iterdir())[:30]:
        print("   ", x.name + ("/" if x.is_dir() else ""))

print()
print("If a candidate above looks right, set DATASET_PATH to it (Cell 7) and rerun.")
print("Common gotcha: some UCF-101 downloads nest as 'UCF-101/UCF-101/<class folders>' --")
print("if so, point DATASET_PATH at the INNER folder, the one whose subfolders are the")
print("101 class names directly (ApplyEyeMakeup, ApplyLipstick, Archery, ...).")


In [ ]:
# Load dataset — first run builds disk cache, subsequent runs load from it
dataset = UCF11VideoDataset(DATASET_PATH,
                             sequence_length=SEQ_LEN,
                             target_size=INPUT_SIZE,
                             sample_fps=SAMPLE_FPS,
                             augment=False,
                             cache_dir=CACHE_DIR,
                             class_subset=CLASS_SUBSET)
NUM_CLASSES = len(dataset.class_to_idx)
CLASS_NAMES = [dataset.idx_to_class[i] for i in range(NUM_CLASSES)]
print('Classes ({})  : {}'.format(NUM_CLASSES, CLASS_NAMES))
print('Total videos  :', len(dataset))


## Cell 8B — ResNet-18 / MobileNetV2 Baselines

In [ ]:
import torchvision.models as tvm

class ResNet18Baseline(nn.Module):
    """Table 3: 'ResNet-18' row. Frame-wise ResNet-18 (from scratch, no
    pretrained weights, to match SAM-CapsNet's no-pretrained-backbone protocol),
    temporal-mean-pooled logits over T frames."""
    def __init__(self, num_classes=11):
        super().__init__()
        m = tvm.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.backbone = m

    def forward(self, x):
        B, T, C, H, W = x.shape
        logits = self.backbone(x.view(B * T, C, H, W))
        return logits.view(B, T, -1).mean(dim=1)


class MobileNetV2Baseline(nn.Module):
    """Table 3: 'MobileNetV2' row. Same protocol as ResNet18Baseline."""
    def __init__(self, num_classes=11):
        super().__init__()
        m = tvm.mobilenet_v2(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        self.backbone = m

    def forward(self, x):
        B, T, C, H, W = x.shape
        logits = self.backbone(x.view(B * T, C, H, W))
        return logits.view(B, T, -1).mean(dim=1)


print("ResNet18Baseline and MobileNetV2Baseline defined.")
_d = torch.zeros(2, 16, 3, 224, 224)
for _cls in (ResNet18Baseline, MobileNetV2Baseline):
    _m = _cls(num_classes=11)
    with torch.no_grad():
        _o = _m(_d)
    assert tuple(_o.shape) == (2, 11), _cls.__name__
    print("  {:<20} forward OK: {}".format(_cls.__name__, tuple(_o.shape)))
del _d, _m, _o

# ---- Train both, 5 seeds each, on the 11-class dataset loaded via Cell 7A/8 ----
resnet_mobilenet_results = {}
for name, ctor in [('ResNet-18', ResNet18Baseline), ('MobileNetV2', MobileNetV2Baseline)]:
    seed_test = []
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    for seed in SEEDS:
        r = run_one_seed_logits(ctor, dataset, NUM_CLASSES, seed,
                                NUM_EPOCHS, BATCH_SIZE, LR, DEVICE)
        seed_test.append(r['test_acc'])
        print("  Seed {}: test = {:.2f}%".format(seed, r['test_acc']))
    mean_t, std_t = float(np.mean(seed_test)), float(np.std(seed_test))
    n_params = sum(p.numel() for p in ctor(NUM_CLASSES).parameters())
    resnet_mobilenet_results[name] = dict(
        mean_test=mean_t, std_test=std_t, seed_test=seed_test,
        n_params=n_params)
    print("  Result: {:.2f} +/- {:.2f}%   Params: {:.2f}M".format(
          mean_t, std_t, n_params / 1e6))

with open(SAVE_DIR + '/resnet_mobilenet_baselines.json', 'w') as f:
    json.dump(resnet_mobilenet_results, f, indent=2)
print("\nSaved:", SAVE_DIR + '/resnet_mobilenet_baselines.json')


## Cell 8C — Parameter Counts (Table 3 "Params" column)


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print("Parameter counts (as actually implemented in this notebook):\n")

for label, n_cls, in_size, p_caps in [
    ("SAM-CapsNet, 11-class config (Cell 7A defaults)",  11,  INPUT_SIZE if 'CLASS_SUBSET' in dir() and CLASS_SUBSET else (224,224), 32),
    ("SAM-CapsNet, 101-class config (112x112, 16 caps)", 101, (112, 112), 16),
]:
    m = SAMCapsNet(num_classes=n_cls, sam_config={}, input_size=in_size, prim_caps=p_caps)
    n = count_params(m)
    print("  {:<45}: {:>10,} params ({:.2f}M)".format(label, n, n / 1e6))
    del m

for label, ctor, n_cls in [
    ("Baseline CNN (softmax)", BaselineCNN, 11),
    ("CNN + Multi-Scale SAM",  CNNSAMNoCapsule, 11),
    ("ResNet-18",              ResNet18Baseline, 11),
    ("MobileNetV2",            MobileNetV2Baseline, 11),
]:
    m = ctor(n_cls)
    n = count_params(m)
    print("  {:<45}: {:>10,} params ({:.2f}M)".format(label, n, n / 1e6))
    del m


## Cell 8D — Frame-Count (T) Grid Search


In [ ]:
from copy import deepcopy

t_grid_results = {}
_, full_cfg, _, _ = CONFIG_BY_KEY['full_sam']

for T in [8, 12, 16, 24]:
    print("\n" + "=" * 50)
    print("T =", T)
    print("=" * 50)
    ds_T = UCF11VideoDataset(DATASET_PATH, sequence_length=T,
                             target_size=INPUT_SIZE, sample_fps=SAMPLE_FPS,
                             augment=False, cache_dir=CACHE_DIR + '_T{}'.format(T),
                             class_subset=CLASS_SUBSET)
    r = run_one_seed(ds_T, NUM_CLASSES, full_cfg, seed=42,
                     num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                     device=DEVICE, input_size=INPUT_SIZE, prim_caps=PRIM_CAPS)
    t_grid_results[T] = dict(val_acc=r['best_val'], test_acc=r['test_acc'])
    print("  T={}: best val = {:.2f}%   test = {:.2f}%".format(
          T, r['best_val'], r['test_acc']))
    del ds_T

print("\nGrid search summary:")
for T, r in t_grid_results.items():
    print("  T={:<3}: val {:.2f}%  test {:.2f}%".format(T, r['val_acc'], r['test_acc']))
best_T = max(t_grid_results, key=lambda t: t_grid_results[t]['val_acc'])
print("\nBest T by validation accuracy:", best_T)

with open(SAVE_DIR + '/t_grid_search.json', 'w') as f:
    json.dump(t_grid_results, f, indent=2)


## Cell 9 — Quick Single Run

Run this first to verify everything works before the full ablation.

In [ ]:
# Quick single run — full SAM-CapsNet, seed 42
# Use this to verify the pipeline before running the full ablation.
print("Running quick check: SAM-CapsNet (full), seed=42 ...")
_, full_cfg, _, _ = CONFIG_BY_KEY['full_sam']

quick_r = run_one_seed(dataset, NUM_CLASSES, full_cfg,
                       seed=42, num_epochs=NUM_EPOCHS,
                       batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
                       input_size=INPUT_SIZE, prim_caps=PRIM_CAPS)

print("")
print("Test accuracy : {:.2f}%".format(quick_r['test_acc']))
print("Best val acc  : {:.2f}%".format(quick_r['best_val']))

plot_training_curves(quick_r['train_accs'], quick_r['val_accs'],
                     title='SAM-CapsNet (quick)',
                     save_path=SAVE_DIR + '/quick_curves.png')
plot_confusion_matrix(quick_r['gt'], quick_r['preds'], CLASS_NAMES,
                      title='SAM-CapsNet (quick)',
                      save_path=SAVE_DIR + '/quick_confusion.png')


## Cell 9B — Proposed-Only Run on Full UCF-101 (5 seeds)

Runs just `full_sam` (SAM-CapsNet, the proposed config) across all 5 seeds on the
full 101-class dataset.

Run Cell 7B and Cell 8 (with `CLASS_SUBSET = None`) before this cell.


In [ ]:
# Proposed-only run: SAM-CapsNet (full_sam), 5 seeds, full UCF-101 (101 classes)
_, full_cfg, full_label, _ = CONFIG_BY_KEY['full_sam']

proposed_ucf101_results = {'seed_test': [], 'seed_val': [], 'preds': [], 'gt': []}

for s_idx, seed in enumerate(SEEDS):
    print("Seed {} ({}/{}):".format(seed, s_idx + 1, len(SEEDS)))
    r = run_one_seed(dataset, NUM_CLASSES, full_cfg, seed,
                      NUM_EPOCHS, BATCH_SIZE, LR, DEVICE,
                      input_size=INPUT_SIZE, prim_caps=PRIM_CAPS)
    proposed_ucf101_results['seed_test'].append(r['test_acc'])
    proposed_ucf101_results['seed_val'].append(r['best_val'])
    proposed_ucf101_results['preds'].extend(r['preds'].tolist())
    proposed_ucf101_results['gt'].extend(r['gt'].tolist())
    print("  -> Seed {}: test = {:.2f}%   best val = {:.2f}%".format(
        seed, r['test_acc'], r['best_val']))

proposed_ucf101_results['model'] = r['model']   # keep last-seed model for latency benchmarking (Cell 9C)

mean_t = float(np.mean(proposed_ucf101_results['seed_test']))
std_t  = float(np.std(proposed_ucf101_results['seed_test']))
mean_v = float(np.mean(proposed_ucf101_results['seed_val']))
std_v  = float(np.std(proposed_ucf101_results['seed_val']))

print("")
print("=" * 60)
print("SAM-CapsNet (Proposed) -- full UCF-101, {} classes, {} seeds".format(
    NUM_CLASSES, len(SEEDS)))
print("Test accuracy : {:.2f} +/- {:.2f}%".format(mean_t, std_t))
print("Best val acc  : {:.2f} +/- {:.2f}%".format(mean_v, std_v))
print("Per-seed test :", ["{:.2f}".format(x) for x in proposed_ucf101_results['seed_test']])
print("=" * 60)

# Save so Table 3 can be updated from this file without retraining
summary = {
    'config': 'full_sam',
    'label': 'SAM-CapsNet (Proposed)',
    'dataset': 'UCF-101 (101 classes)',
    'num_classes': NUM_CLASSES,
    'seeds': SEEDS,
    'seed_test_acc': proposed_ucf101_results['seed_test'],
    'seed_val_acc': proposed_ucf101_results['seed_val'],
    'mean_test_acc': mean_t,
    'std_test_acc': std_t,
    'mean_val_acc': mean_v,
    'std_val_acc': std_v,
}
with open(SAVE_DIR + '/proposed_ucf101_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Saved:", SAVE_DIR + '/proposed_ucf101_summary.json')

plot_confusion_matrix(np.array(proposed_ucf101_results['gt']),
                       np.array(proposed_ucf101_results['preds']),
                       CLASS_NAMES, title='SAM-CapsNet (Proposed) - UCF-101 (101 classes)',
                       save_path=SAVE_DIR + '/proposed_ucf101_confusion.png')


## Cell 9C — Inference Latency Benchmark

Measures per-clip inference latency (forward pass only, not video decode/preprocessing)
for whichever trained model is currently in memory, using random dummy input at the
model's actual input shape (`SEQ_LEN`, `INPUT_SIZE` from Cell 7) so results are
directly comparable across configs/datasets as long as you re-run this after
switching configs.

Run this:
- after Cell 9 (quick run) to benchmark on whatever config Cell 7 was set to then, or
- after Cell 9B to benchmark the proposed model on full UCF-101 (what Table 3 needs
  for the `CapsNet+SAM` / UCF-101 row).

In [ ]:
import time

def benchmark_latency(model, device, seq_len, input_size,
                       num_warmup=10, num_runs=100, batch_size=1):
    """
    Per-clip inference latency, forward pass only.
    Uses random dummy input at the model's real input shape -- this measures
    architecture cost, not video decode/IO/preprocessing time, consistent with
    how inference latency is usually reported in the literature. CUDA calls are
    synchronised before/after each timed pass so GPU work is fully captured.
    """
    model.eval()
    H, W = input_size
    dummy = torch.randn(batch_size, seq_len, 3, H, W, device=device)

    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        times_ms = []
        for _ in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(dummy)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000.0)

    times_ms = np.array(times_ms)
    mean_ms, std_ms = float(times_ms.mean()), float(times_ms.std())
    clips_per_sec = 1000.0 / mean_ms * batch_size

    print("Latency benchmark: {} timed passes, batch={}, device={}".format(
        num_runs, batch_size, device))
    print("  Per-clip latency : {:.2f} +/- {:.2f} ms".format(mean_ms, std_ms))
    print("  Throughput       : {:.1f} clips/s".format(clips_per_sec))

    return dict(mean_ms=mean_ms, std_ms=std_ms, clips_per_sec=clips_per_sec,
                num_warmup=num_warmup, num_runs=num_runs, batch_size=batch_size,
                seq_len=seq_len, input_size=list(input_size), device=str(device))


# Prefer the model just trained in Cell 9B (proposed, full UCF-101); fall back
# to the Cell 9 quick-run model if 9B has not been run yet.
_bench_model = None
if 'proposed_ucf101_results' in dir() and proposed_ucf101_results.get('model') is not None:
    _bench_model = proposed_ucf101_results['model']
    _bench_tag   = 'proposed_ucf101'
elif 'quick_r' in dir():
    _bench_model = quick_r['model']
    _bench_tag   = 'quick_run'
else:
    raise RuntimeError("No trained model in memory -- run Cell 9 or Cell 9B first.")

latency_summary = benchmark_latency(_bench_model, DEVICE, SEQ_LEN, INPUT_SIZE,
                                     num_warmup=10, num_runs=100, batch_size=1)
latency_summary['tag'] = _bench_tag
latency_summary['num_classes'] = NUM_CLASSES

out_path = SAVE_DIR + '/latency_benchmark_{}.json'.format(_bench_tag)
with open(out_path, 'w') as f:
    json.dump(latency_summary, f, indent=2)
print("Saved:", out_path)


## Cell 10 — Full Ablation Study

Runs all 8 configurations × `NUM_SEEDS` seeds. Saves JSON after each config so progress is preserved on crash.

> **Tip:** Set `NUM_SEEDS=2` and `NUM_EPOCHS=10` in Cell 7 for a fast test run first.

## Cell 10A — Subject-Independent Split


In [ ]:
import re

def make_subject_independent_split(dataset, test_groups=None, seed=42):
    """
    Leave-groups-out split for UCF-11.

    UCF-11 encodes subject groups in TWO ways depending on distribution:
      1. Filename: v_Basketball_g01_c01.mpg  -> group 'g01'
      2. Subfolder: Basketball/group01/clip001.mpg -> group 'group01'

    This function checks BOTH the filename AND the parent folder name
    so it works with any UCF-11 distribution layout.
    """
    file_re   = re.compile(r'_g(\d{2})_')       # v_Class_g01_c01
    folder_re = re.compile(r'group(\d+)', re.I)  # group01/ or Group01/

    group_to_indices = {}
    ungrouped = []

    for i, (path, _) in enumerate(dataset.samples):
        from pathlib import Path as _P
        p = _P(path)

        # Try filename first
        m = file_re.search(p.name)
        if m:
            g = 'g{:02d}'.format(int(m.group(1)))
        else:
            # Try parent folder name
            m2 = folder_re.search(p.parent.name)
            if m2:
                g = 'g{:02d}'.format(int(m2.group(1)))
            else:
                # Try grandparent folder (some layouts nest deeper)
                m3 = folder_re.search(p.parent.parent.name)
                if m3:
                    g = 'g{:02d}'.format(int(m3.group(1)))
                else:
                    ungrouped.append(i)
                    continue

        group_to_indices.setdefault(g, []).append(i)

    all_groups = sorted(group_to_indices.keys())
    print('Subject groups found: {}'.format(all_groups))

    if len(all_groups) == 0:
        print('WARNING: No subject groups detected in filenames or folder names.')
        print('         Falling back to stratified random split.')
        import numpy as np
        from sklearn.model_selection import StratifiedShuffleSplit
        labels = np.array(dataset.get_labels())
        sss = StratifiedShuffleSplit(1, test_size=0.15, random_state=seed)
        train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
        return train_idx.tolist(), test_idx.tolist()

    if test_groups is None:
        import random as _random
        rng = _random.Random(seed)
        n_test = max(1, int(len(all_groups) * 0.15))
        test_groups = rng.sample(all_groups, n_test)

    print('Held-out test groups: {}'.format(test_groups))
    train_idx, test_idx = [], []
    for g, idxs in group_to_indices.items():
        (test_idx if g in test_groups else train_idx).extend(idxs)
    train_idx.extend(ungrouped)
    print('Subject-independent split -> train: {}, test: {}'.format(
          len(train_idx), len(test_idx)))
    return train_idx, test_idx


def run_subject_independent(dataset, num_classes, sam_config,
                             num_epochs, batch_size, lr, device):
    import numpy as np, math, torch, torch.nn as nn
    from torch.utils.data import DataLoader, Subset

    seed_accs = []
    for seed in SEEDS:
        set_seed(seed)
        train_idx, test_idx = make_subject_independent_split(dataset, seed=seed)

        if len(test_idx) == 0:
            print('  Seed {}: no test samples after split, skipping.'.format(seed))
            continue

        n_val = max(1, int(len(train_idx) * 0.15))
        val_idx   = train_idx[:n_val]
        train_idx = train_idx[n_val:]

        train_ld = DataLoader(Subset(dataset, train_idx), batch_size=batch_size,
                              shuffle=True,  num_workers=0)
        val_ld   = DataLoader(Subset(dataset, val_idx),   batch_size=batch_size,
                              shuffle=False, num_workers=0)
        test_ld  = DataLoader(Subset(dataset, test_idx),  batch_size=batch_size,
                              shuffle=False, num_workers=0)

        model = SAMCapsNet(num_classes=num_classes, sam_config=sam_config).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

        def lr_lam(ep):
            warmup = 3
            if ep < warmup:
                return (ep + 1) / warmup
            return 0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(num_epochs - warmup, 1)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lam)
        best_val, best_state = 0.0, None

        for ep in range(num_epochs):
            tr_loss, tr_acc = train_one_epoch(model, train_ld, opt, device)
            val_acc, _, _   = evaluate(model, val_ld, device)
            scheduler.step()
            if val_acc > best_val:
                best_val   = val_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            print('  [SI seed={}] Ep {:02d} | train {:.1f}% | val {:.1f}%'.format(
                  seed, ep + 1, tr_acc, val_acc))

        model.load_state_dict(best_state)
        test_acc, _, _ = evaluate(model, test_ld, device)
        seed_accs.append(test_acc)
        print('  [SI seed={}] test: {:.2f}%'.format(seed, test_acc))

    if not seed_accs:
        print('No valid seeds completed.')
        return 0.0, 0.0

    import numpy as np
    m, s = float(np.mean(seed_accs)), float(np.std(seed_accs))
    print('\nSubject-independent result: {:.2f} +/- {:.2f}%'.format(m, s))
    return m, s


# Run it
_, full_sam_cfg, _, _ = CONFIG_BY_KEY['full_sam']
si_mean, si_std = run_subject_independent(
    dataset, NUM_CLASSES, full_sam_cfg, NUM_EPOCHS, BATCH_SIZE, LR, DEVICE)


In [ ]:
# Full ablation: 8 configurations x NUM_SEEDS seeds
# Saves per-class accuracy for every config so we can compare across all ablations.

configs_to_run = [c[0] for c in ABLATION_CONFIGS]
all_results    = {}
trained_models = {}   # NEW: keep the last-seed trained model per config in memory
                       # (not JSON-serialisable, so kept separate from all_results) --
                       # needed by Cells 15-19 below for the capsule-space
                       # interpretability analyses (Sec. II-E, Figs. 2-4).
last_run_preds = {}    # NEW: last-seed (preds, gt) per config, for single-run
last_run_gt    = {}    # comparisons like Fig. 5's confusion matrix ("on this run").

for config_key in configs_to_run:
    _, sam_cfg, label, desc = CONFIG_BY_KEY[config_key]
    print("")
    print("=" * 65)
    print("CONFIG: {}".format(label))
    print("  {}".format(desc))
    print("=" * 65)

    seed_test, all_preds, all_gt = [], [], []

    for s_idx, seed in enumerate(SEEDS):
        print("  Seed {} ({}/{}):".format(seed, s_idx + 1, len(SEEDS)))
        r = run_one_seed(dataset, NUM_CLASSES, sam_cfg, seed,
                         NUM_EPOCHS, BATCH_SIZE, LR, DEVICE,
                         input_size=INPUT_SIZE, prim_caps=PRIM_CAPS)
        seed_test.append(r['test_acc'])
        all_preds.extend(r['preds'].tolist())
        all_gt.extend(r['gt'].tolist())
        print("  -> Seed {}: test = {:.2f}%".format(seed, r['test_acc']))

        if s_idx == len(SEEDS) - 1:
            plot_training_curves(r['train_accs'], r['val_accs'],
                                 title=label,
                                 save_path=SAVE_DIR + '/' + config_key + '_curves.png')
            plot_confusion_matrix(r['gt'], r['preds'], CLASS_NAMES,
                                  title=label,
                                  save_path=SAVE_DIR + '/' + config_key + '_confusion.png')
            trained_models[config_key] = r['model']   # NEW: stash last-seed model
            last_run_preds[config_key]  = r['preds']    # NEW: stash last-seed preds
            last_run_gt[config_key]     = r['gt']       # NEW: stash last-seed gt

    mean_t, std_t = np.mean(seed_test), np.std(seed_test)

    # Compute per-class accuracy pooled across all seeds x test folds
    all_preds_arr = np.array(all_preds)
    all_gt_arr    = np.array(all_gt)
    per_class_acc = {}
    for ci, cn in enumerate(CLASS_NAMES):
        mask = all_gt_arr == ci
        if mask.sum() > 0:
            per_class_acc[cn] = float((all_preds_arr[mask] == ci).mean() * 100)
        else:
            per_class_acc[cn] = 0.0

    print("")
    print("  Result: {:.2f} +/- {:.2f}%".format(mean_t, std_t))
    print("  Per-class:")
    for cn, acc in per_class_acc.items():
        print("    {:<22}: {:.1f}%".format(cn, acc))

    # NEW: macro-averaged precision/recall/F1 (paper's Table 1 columns 3-5),
    # computed from the pooled preds/gt across all seeds' test folds.
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        all_gt_arr, all_preds_arr, average='macro', zero_division=0)

    all_results[config_key] = dict(
        label=label, description=desc,
        test_accs_per_seed=seed_test,
        mean_test=mean_t, std_test=std_t,
        per_class_acc=per_class_acc,
        precision_macro=float(prec_m),
        recall_macro=float(rec_m),
        f1_macro=float(f1_m),
        all_preds=all_preds,   # NEW: pooled predictions (JSON-safe list)
        all_gt=all_gt,         # NEW: pooled ground-truth labels (JSON-safe list)
    )
    with open(SAVE_DIR + '/ablation_table.json', 'w') as f:
        json.dump(all_results, f, indent=2)

# Summary table
print("")
print("=" * 65)
print("ABLATION RESULTS")
print("=" * 65)
print("{:<28}  {}".format("Config", "Test Acc (%)"))
print("-" * 65)
for key, r in all_results.items():
    mark = "  <- proposed" if key == "full_sam" else ""
    print("{:<28}  {:.2f} +/- {:.2f}%{}".format(
          r['label'], r['mean_test'], r['std_test'], mark))
print("=" * 65)

plot_ablation_bar(all_results,      save_path=SAVE_DIR + '/ablation_bar.png')
plot_component_heatmap(all_results, save_path=SAVE_DIR + '/component_heatmap.png')


## Cell 10B — Baseline CNN / CNN+SAM Training Runs (Table 1, rows 1–2)

Trains `BaselineCNN` and `CNNSAMNoCapsule` (Cell 4B) for 5 seeds each on the
11-class dataset, using the same logits training loop as the ResNet/MobileNetV2
baselines. Together with the 8-config capsule ablation (Cell 10, `all_results`),
this gives all 10 rows needed to reassemble Table 1 (Cell 10C).

Run after Cell 7A + Cell 8 (11-class dataset loaded), and after Cell 10 has
populated `all_results` (or run independently — this cell doesn't depend on it).


In [ ]:
baseline_results   = {}
baseline_models    = {}   # NEW: last-seed model per baseline (for Fig. 5 / Sec. III-C)
baseline_last_preds = {}  # NEW: last-seed preds/gt per baseline (single-run comparisons)
baseline_last_gt    = {}
for key, label, ctor in [
    ('baseline_cnn', 'Baseline CNN (softmax)', BaselineCNN),
    ('cnn_sam',      'CNN + Multi-Scale SAM',  CNNSAMNoCapsule),
]:
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    seed_test, all_preds, all_gt = [], [], []
    r = None
    for seed in SEEDS:
        r = run_one_seed_logits(ctor, dataset, NUM_CLASSES, seed,
                                NUM_EPOCHS, BATCH_SIZE, LR, DEVICE)
        seed_test.append(r['test_acc'])
        all_preds.extend(np.asarray(r['preds']).tolist())   # NEW: pool preds
        all_gt.extend(np.asarray(r['gt']).tolist())          # NEW: pool gt
        print("  Seed {}: test = {:.2f}%".format(seed, r['test_acc']))
    baseline_models[key]     = r['model']   # NEW: stash last-seed model
    baseline_last_preds[key] = r['preds']
    baseline_last_gt[key]    = r['gt']

    mean_t, std_t = float(np.mean(seed_test)), float(np.std(seed_test))

    # NEW: macro precision/recall/F1 + pooled preds/gt, matching the ablation
    # loop above (Table 1 needs these columns for every row, not just the
    # 8 capsule configs).
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        np.array(all_gt), np.array(all_preds), average='macro', zero_division=0)

    baseline_results[key] = dict(label=label, mean_test=mean_t, std_test=std_t,
                                 seed_test=seed_test,
                                 precision_macro=float(prec_m),
                                 recall_macro=float(rec_m),
                                 f1_macro=float(f1_m),
                                 all_preds=all_preds, all_gt=all_gt)
    print("  Result: {:.2f} +/- {:.2f}%".format(mean_t, std_t))

    # NEW: last-seed confusion matrix for this baseline (Fig. 5 needs the
    # CNN + Multi-Scale SAM confusion matrix specifically, to compare against
    # SAM-CapsNet's).
    plot_confusion_matrix(r['gt'], r['preds'], CLASS_NAMES, title=label,
                          save_path=SAVE_DIR + '/' + key + '_confusion.png')

with open(SAVE_DIR + '/baseline_cnn_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print("\nSaved:", SAVE_DIR + '/baseline_cnn_results.json')


## Cell 10C — Assemble Table 1

Combines the 8-config capsule ablation (`all_results`, Cell 10) with the two
baseline runs (`baseline_results`, Cell 10B) into the full Table 1, in the paper's
row order. Run after both Cell 10 and Cell 10B have populated their result dicts.


In [ ]:
row_order = [
    ('baseline_cnn', None),
    ('cnn_sam',      None),
    ('no_sam',       'ablation'),
    ('full_sam',     'ablation'),
    ('ch_only',      'ablation'),
    ('tm_only',      'ablation'),
    ('sp_only',      'ablation'),
    ('ch_tm',        'ablation'),
    ('tm_sp',        'ablation'),
    ('ch_sp',        'ablation'),
]

# NEW: full Table 1 replication including Precision/Recall/F1 (Macro), which
# the notebook previously never computed (only accuracy). Values come from
# precision_recall_fscore_support(average='macro') over pooled preds/gt,
# added to all_results / baseline_results in Cells 10 and 10B above.
header = "{:<28} {:>12} {:>12} {:>12} {:>12}".format(
    "Configuration", "Acc (%)", "Prec(M)", "Rec(M)", "F1(M)")
print(header)
print("-" * len(header))
table1_rows = []
for key, source in row_order:
    r = (all_results[key] if source == 'ablation' else baseline_results[key])
    print("{:<28} {:>7.2f}+/-{:<4.2f} {:>12.4f} {:>12.4f} {:>12.4f}".format(
          r['label'], r['mean_test'], r['std_test'],
          r['precision_macro'], r['recall_macro'], r['f1_macro']))
    table1_rows.append(dict(configuration=r['label'],
                            accuracy_mean=r['mean_test'], accuracy_std=r['std_test'],
                            precision_macro=r['precision_macro'],
                            recall_macro=r['recall_macro'],
                            f1_macro=r['f1_macro']))
print("-" * len(header))
print("Compare each row against Table 1 of the submitted paper.")

with open(SAVE_DIR + '/table1_full.json', 'w') as f:
    json.dump(table1_rows, f, indent=2)
print("Saved:", SAVE_DIR + '/table1_full.json')


## Cell 11 — Per-Class Accuracy Across All Ablation Configurations

Three outputs:

1. **Text table** — rows = classes, columns = configs, easy to copy into paper
2. **Heatmap** — colour-coded grid, proposed column highlighted in orange
3. **Grouped bar chart** — one group per class, all configs side by side

Runs automatically from `all_results` (Cell 10) or loads from saved JSON.

In [ ]:
# Per-class accuracy table and heatmap across all ablation configurations.
# Shows for every class (e.g. Walking, Basketball) what % accuracy each
# ablation config achieved — side by side for direct comparison.

def plot_per_class_heatmap(results, class_names, save_path=None):
    """
    Heatmap: rows = activity classes, columns = ablation configs.
    Cell value = per-class accuracy (%).
    Proposed config column is highlighted in orange.
    """
    config_keys   = list(results.keys())
    config_labels = [results[k]['label'] for k in config_keys]
    n_classes     = len(class_names)
    n_configs     = len(config_keys)

    # Build matrix: rows = classes, cols = configs
    matrix = np.zeros((n_classes, n_configs))
    for j, key in enumerate(config_keys):
        pc = results[key].get('per_class_acc', {})
        for i, cn in enumerate(class_names):
            matrix[i, j] = pc.get(cn, 0.0)

    fig, ax = plt.subplots(figsize=(max(14, n_configs * 1.6),
                                    max(6, n_classes * 0.65)))

    # Colour map: white=0%, dark blue=100%
    im = ax.imshow(matrix, cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')
    plt.colorbar(im, ax=ax, label='Per-class accuracy (%)', shrink=0.8)

    ax.set_xticks(range(n_configs))
    ax.set_xticklabels(config_labels, rotation=35, ha='right', fontsize=10)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(class_names, fontsize=10)

    # Annotate each cell with the value
    for i in range(n_classes):
        for j in range(n_configs):
            val = matrix[i, j]
            txt_col = 'white' if val > 65 else 'black'
            ax.text(j, i, '{:.0f}'.format(val),
                    ha='center', va='center', fontsize=8.5,
                    color=txt_col, fontweight='bold')

    # Highlight the proposed column with an orange border
    if 'full_sam' in config_keys:
        proposed_col = config_keys.index('full_sam')
        for i in range(n_classes):
            ax.add_patch(plt.Rectangle(
                (proposed_col - 0.5, i - 0.5), 1, 1,
                linewidth=2.5, edgecolor='#FF5722', facecolor='none', zorder=3))

    ax.set_title('Per-Class Accuracy (%) Across All Ablation Configurations',
                 fontsize=13, fontweight='bold', pad=14)
    ax.set_xlabel('Ablation Configuration', fontsize=11)
    ax.set_ylabel('Activity Class', fontsize=11)

    # Add column mean (overall accuracy) at the bottom
    ax.axhline(n_classes - 0.5, color='gray', lw=0.8, ls='--')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def print_per_class_table(results, class_names):
    """
    Prints a readable text table:
    Class | Proposed | Pure CapsNet | Channel | ...
    """
    config_keys   = list(results.keys())
    config_labels = [results[k]['label'] for k in config_keys]

    # Header
    col_w = 14
    header = "{:<22}".format("Class")
    for lbl in config_labels:
        header += "  {:>{}}".format(lbl[:col_w], col_w)
    print("=" * len(header))
    print("PER-CLASS ACCURACY (%) — ALL ABLATION CONFIGURATIONS")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for cn in class_names:
        row = "{:<22}".format(cn)
        for key in config_keys:
            pc  = results[key].get('per_class_acc', {})
            val = pc.get(cn, 0.0)
            row += "  {:>{}.1f}".format(val, col_w)
        print(row)

    # Overall mean row
    print("-" * len(header))
    row = "{:<22}".format("OVERALL (mean)")
    for key in config_keys:
        row += "  {:>{}.2f}".format(results[key]['mean_test'], col_w)
    print(row)
    print("=" * len(header))


def plot_per_class_grouped_bar(results, class_names, save_path=None):
    """
    Grouped bar chart: one group per activity class,
    one bar per ablation config inside each group.
    Makes it easy to see which classes benefit most from full SAM.
    """
    config_keys   = list(results.keys())
    config_labels = [results[k]['label'] for k in config_keys]
    n_configs     = len(config_keys)
    n_classes     = len(class_names)

    matrix = np.zeros((n_classes, n_configs))
    for j, key in enumerate(config_keys):
        pc = results[key].get('per_class_acc', {})
        for i, cn in enumerate(class_names):
            matrix[i, j] = pc.get(cn, 0.0)

    x     = np.arange(n_classes)
    width = 0.8 / n_configs
    colors = ['#FF5722' if k == 'full_sam' else
              plt.cm.tab10(ci / max(n_configs - 1, 1))
              for ci, k in enumerate(config_keys)]

    fig, ax = plt.subplots(figsize=(max(16, n_classes * 1.4), 7))
    for j in range(n_configs):
        offset = (j - n_configs / 2 + 0.5) * width
        bars = ax.bar(x + offset, matrix[:, j], width * 0.9,
                      label=config_labels[j], color=colors[j], alpha=0.88)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=35, ha='right', fontsize=10)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_ylim(0, 115)
    ax.set_title('Per-Class Accuracy by Ablation Configuration',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8.5, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(100, color='gray', lw=0.8, ls=':')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


# ── Run all three outputs ──────────────────────────────────────────────────────

if 'all_results' in dir() and all_results:
    results_to_use = all_results
elif 'loaded' in dir() and loaded:
    results_to_use = loaded
else:
    # Try loading from JSON
    try:
        with open(SAVE_DIR + '/ablation_table.json') as f:
            results_to_use = json.load(f)
        print("Loaded results from JSON.")
    except FileNotFoundError:
        print("No results found. Run Cell 10 (full ablation) first.")
        results_to_use = None

if results_to_use is not None:
    print_per_class_table(results_to_use, CLASS_NAMES)

    plot_per_class_heatmap(
        results_to_use, CLASS_NAMES,
        save_path=SAVE_DIR + '/per_class_heatmap.png')

    plot_per_class_grouped_bar(
        results_to_use, CLASS_NAMES,
        save_path=SAVE_DIR + '/per_class_grouped_bar.png')


## Cell 10D — Assemble Table 3 (authors-trained rows)

SAM-CapsNet at 11 classes (`all_results['full_sam']`, Cell 10)
and at 101 classes (`proposed_ucf101_results`, Cell 9B), plus ResNet-18 /
MobileNetV2 (`resnet_mobilenet_results`, Cell 8B).

Run after Cell 8B, Cell 9B, and Cell 10 have all populated their result variables.


In [ ]:
def _params(ctor_or_kwargs):
    return sum(p.numel() for p in ctor_or_kwargs.parameters())

rows = []

# SAM-CapsNet, 101 classes (Cell 9B)
m = SAMCapsNet(num_classes=101, sam_config={}, input_size=(112, 112), prim_caps=16)
rows.append(("SAM-CapsNet (Proposed), UCF-101 (101), 16/video",
            _params(m), "{:.2f} +/- {:.2f}%".format(
                float(np.mean(proposed_ucf101_results['seed_test'])),
                float(np.std(proposed_ucf101_results['seed_test'])))))
del m

# SAM-CapsNet, 11 classes (from the ablation dict, Cell 10)
m = SAMCapsNet(num_classes=11, sam_config={}, input_size=INPUT_SIZE, prim_caps=PRIM_CAPS)
r = all_results['full_sam']
rows.append(("SAM-CapsNet (Proposed), 11-class subset, 16/video",
            _params(m), "{:.2f} +/- {:.2f}%".format(r['mean_test'], r['std_test'])))
del m

# ResNet-18 / MobileNetV2 (Cell 8B)
for name in ('ResNet-18', 'MobileNetV2'):
    r = resnet_mobilenet_results[name]
    rows.append((name + ", 11-class subset, 16/video",
                r['n_params'], "{:.2f} +/- {:.2f}%".format(r['mean_test'], r['std_test'])))

print("{:<48} {:>12} {:>18}".format("Method", "Params", "Accuracy"))
print("-" * 80)
for label, n_params, acc in rows:
    print("{:<48} {:>10.2f}M {:>18}".format(label, n_params / 1e6, acc))
print("-" * 80)
print("Compare against the 'authors-trained' rows of Table 3 in the submitted paper.")
print("(All other Table 3 rows are literature figures, not reproducible here.)")


## Cell 12 — Load & Replot Saved Results

In [ ]:
# Load and replot from saved JSON (run without retraining)
import json

with open(SAVE_DIR + '/ablation_table.json') as f:
    loaded = json.load(f)

print("{} configs loaded.".format(len(loaded)))
print("{:<28}  {}".format("Config", "Test Acc (%)"))
print("-" * 60)
for key, r in loaded.items():
    mark = "  <- proposed" if key == "full_sam" else ""
    print("{:<28}  {:.2f} +/- {:.2f}%{}".format(
          r['label'], r['mean_test'], r['std_test'], mark))

plot_ablation_bar(loaded,      save_path=SAVE_DIR + '/ablation_bar.png')
plot_component_heatmap(loaded, save_path=SAVE_DIR + '/component_heatmap.png')


## Cell 13 — Attention + CoM Visualisation

In [ ]:
def visualize_attention_com(model, frames, idx_to_class, device, cmap='jet'):
    model.eval()
    if frames.dim() == 4:
        frames = frames.unsqueeze(0)
    frames = frames.to(device)
    B, T, C, H, W = frames.shape

    with torch.no_grad():
        out, att = model(frames, return_attention=True)
    spatial = att['spatial']

    vecs = out[0].cpu().numpy()
    mags = np.linalg.norm(vecs, axis=1)
    pred = idx_to_class[int(np.argmax(mags))]

    show   = np.linspace(0, T - 1, min(8, T), dtype=int)
    n      = len(show)
    cmap_fn = __import__('matplotlib').cm.get_cmap(cmap)
    trail  = []

    fig, axes = plt.subplots(3, n, figsize=(3 * n, 9))
    if n == 1:
        axes = axes.reshape(-1, 1)

    for col, fi in enumerate(show):
        frame = frames[0, fi].cpu().permute(1, 2, 0).numpy()
        frame = np.clip(frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
        am    = spatial[0, fi, 0].cpu().numpy()
        am    = cv2.resize(am, (W, H), interpolation=cv2.INTER_CUBIC)
        am    = (am - am.min()) / (am.max() - am.min() + 1e-8)

        yg, xg = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        s  = am.sum() + 1e-8
        cx = int((am * xg).sum() / s)
        cy = int((am * yg).sum() / s)
        trail.append((cx, cy))

        att_rgb = cmap_fn(am)[:, :, :3]
        overlay = frame * 0.4 + att_rgb * 0.6

        for row, img in enumerate([frame, att_rgb, overlay]):
            axes[row, col].imshow(np.clip(img, 0, 1))
            axes[row, col].add_patch(plt.Circle((cx, cy), 8, color='red', zorder=5))
            if row == 2 and col > 0:
                px, py = trail[col - 1]
                axes[row, col].plot([px, cx], [py, cy], 'lime', lw=2, ls='--', zorder=4)
            axes[row, col].axis('off')
        axes[0, col].set_title("Frame {}".format(fi + 1), fontsize=8)

    for row, lbl in enumerate(['Original', 'Attention', 'Overlay']):
        axes[row, 0].set_ylabel(lbl, fontsize=9)
    plt.suptitle("Attention + CoM  |  Predicted: {}  (conf={:.3f})".format(
                 pred, mags.max()), fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()


if 'quick_r' in dir() and quick_r['model'] is not None:
    train_idx, val_idx, test_idx = make_stratified_splits(dataset, seed=42)
    sample_idx  = test_idx[0]
    frames_vis, true_lbl = dataset[sample_idx]
    print("Visualising: {} (sample {})".format(
          dataset.idx_to_class[true_lbl], sample_idx))
    visualize_attention_com(quick_r['model'], frames_vis,
                            dataset.idx_to_class, DEVICE)
else:
    print("Run the quick check cell first, then re-run this cell.")


## Cell 14 — Capsule Magnitude & Activation-Attenuation Analysis (Sec. II-E)

This requires `trained_models['no_sam']` and `trained_models['full_sam']`
from Cell 10, and the model's new `return_routing=True` forward path (Cell 4).

In [ ]:
def collect_capsule_magnitudes(model, loader, device):
    """Runs the model with return_routing=True and collects the L2 norm of
    every PrimaryCaps vector and of each sample's winning ActivityCaps
    vector. Matches the paper's Sec. II-E measurement."""
    model.eval()
    primary_norms, activity_norms = [], []
    with torch.no_grad():
        for frames, _ in loader:
            frames = frames.to(device)
            out, extra = model(frames, return_routing=True)
            p_norm = extra['primary'].norm(dim=-1)        # (B, N_primary)
            primary_norms.append(p_norm.flatten().cpu().numpy())
            a_norm = out.norm(dim=-1)                      # (B, num_classes)
            winner = a_norm.max(dim=-1).values              # winning capsule per sample
            activity_norms.append(winner.cpu().numpy())
    return np.concatenate(primary_norms), np.concatenate(activity_norms)


_, _, test_idx_mag = make_stratified_splits(dataset, seed=42)
mag_loader = DataLoader(Subset(dataset, test_idx_mag), batch_size=4,
                        shuffle=False, num_workers=1, pin_memory=True)

attenuation_summary = {}
for key, nice_name in [('no_sam', 'Pure CapsNet (no SAM)'), ('full_sam', 'SAM-CapsNet')]:
    if key not in trained_models:
        print("Skipping {} -- run Cell 10 (full ablation) first.".format(nice_name))
        continue
    p_norms, a_norms = collect_capsule_magnitudes(trained_models[key], mag_loader, DEVICE)
    p_mean, p_std = float(p_norms.mean()), float(p_norms.std())
    a_mean, a_std = float(a_norms.mean()), float(a_norms.std())
    attenuation_pct = 100.0 * (p_mean - a_mean) / p_mean if p_mean > 0 else float('nan')
    attenuation_summary[key] = dict(
        label=nice_name, primary_mean=p_mean, primary_std=p_std,
        activity_mean=a_mean, activity_std=a_std, attenuation_pct=attenuation_pct)
    print("{:<24}  PrimaryCaps {:.3f} +/- {:.3f}   ActivityCaps {:.3f} +/- {:.3f}   attenuation {:.1f}%".format(
          nice_name, p_mean, p_std, a_mean, a_std, attenuation_pct))

with open(SAVE_DIR + '/capsule_attenuation.json', 'w') as f:
    json.dump(attenuation_summary, f, indent=2)
print("Saved:", SAVE_DIR + '/capsule_attenuation.json')
print("Paper (Sec. II-E): Pure CapsNet 0.312+/-0.089 -> 0.241+/-0.071 (22.7% attenuation);")
print("                    SAM-CapsNet 0.518+/-0.063 -> 0.487+/-0.058 (6.0% attenuation).")


## Cell 15 — Coupling-Coefficient Clustering & Peak Confidence (Fig. 2)


(`b_ij`/`c_ij`) or computed a peak-confidence figure. Uses the
`extra['c']` and `extra['s']`/output vectors now exposed by `return_routing=True`.

In [ ]:
def collect_routing_by_class(model, loader, device, num_classes):
    """For each test sample, keeps the final coupling-coefficient column for
    the model's predicted class (how strongly every primary capsule routed to
    the winning output capsule) and that capsule's 16-d instantiation vector,
    grouped by predicted class."""
    model.eval()
    coupling_by_class = {c: [] for c in range(num_classes)}
    vector_by_class    = {c: [] for c in range(num_classes)}
    confidences        = []
    with torch.no_grad():
        for frames, labels in loader:
            frames = frames.to(device)
            out, extra = model(frames, return_routing=True)
            c_coef = extra['c'].squeeze(-1)          # (B, N_primary, num_classes)
            mags   = out.norm(dim=-1)                # (B, num_classes)
            preds  = mags.argmax(dim=-1)
            confidences.extend(mags.max(dim=-1).values.cpu().numpy().tolist())
            for i in range(frames.size(0)):
                pc = int(preds[i].item())
                coupling_by_class[pc].append(c_coef[i, :, pc].cpu().numpy())
                vector_by_class[pc].append(out[i, pc].cpu().numpy())
    return coupling_by_class, vector_by_class, np.array(confidences)


if 'full_sam' in trained_models:
    coupling_by_class, vector_by_class, confidences = collect_routing_by_class(
        trained_models['full_sam'], mag_loader, DEVICE, NUM_CLASSES)

    peak_confidence = float(confidences.mean())
    print("Mean peak classification confidence: {:.3f}  (paper reports 0.753)".format(peak_confidence))

    class_ids = sorted(k for k, v in coupling_by_class.items() if len(v) > 0)
    profiles  = np.stack([np.mean(coupling_by_class[c], axis=0) for c in class_ids])
    coup_pca  = PCA(n_components=2).fit_transform(profiles)

    vectors  = np.stack([np.mean(vector_by_class[c], axis=0) for c in class_ids])
    vec_pca  = PCA(n_components=2).fit_transform(vectors)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    colors = plt.cm.tab20(np.linspace(0, 1, len(class_ids)))

    for i, c in enumerate(class_ids):
        ax1.scatter(*coup_pca[i], s=140, color=colors[i], edgecolor='k', zorder=3)
        ax1.annotate(CLASS_NAMES[c], coup_pca[i], fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax1.set_title("(a) Coupling-coefficient clustering\n(peak confidence {:.3f})".format(peak_confidence))
    ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2"); ax1.grid(alpha=0.3)

    max_r = np.abs(vec_pca).max()
    for i, c in enumerate(class_ids):
        ax2.arrow(0, 0, vec_pca[i, 0], vec_pca[i, 1],
                 head_width=0.03 * max_r, color=colors[i], length_includes_head=True)
        ax2.annotate(CLASS_NAMES[c], vec_pca[i], fontsize=8)
    ax2.axhline(0, color='gray', lw=0.5); ax2.axvline(0, color='gray', lw=0.5)
    ax2.set_title("(b) Capsule orientation patterns per class")
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2"); ax2.grid(alpha=0.3)

    plt.suptitle("FIGURE 2 (reproduced) -- coupling & orientation at convergence", fontweight='bold')
    plt.tight_layout()
    plt.savefig(SAVE_DIR + '/figure2_coupling_orientation.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
else:
    print("Run Cell 10 (full ablation) first so trained_models['full_sam'] exists.")


## Cell 16 — Four-Part Interpretability Figure (Fig. 3): Attention, PrimaryCaps, PCA Separability, Polar View

Panels (b)-(d) — PrimaryCaps formation, PCA capsule-space separability with silhouette scores (paper: 0.641 / 0.487 / 0.219), and the polar confidence view — had no code at all.

In [ ]:
def get_capsule_embeddings(model, loader, device):
    """One embedding per sample: the TRUE class's capsule vector (16-d) for
    capsule models, or the raw logits (num_classes-d) for scalar CNN
    baselines. Used for the PCA/silhouette separability analysis (Fig. 3c)."""
    model.eval()
    embeds, labels = [], []
    with torch.no_grad():
        for frames, y in loader:
            frames = frames.to(device)
            out = model(frames)
            if out.dim() == 3:            # capsule model: (B, num_classes, dim)
                idx  = y.to(device)
                vecs = out[torch.arange(out.size(0)), idx]
            else:                          # logits model: (B, num_classes)
                vecs = out
            embeds.append(vecs.cpu().numpy())
            labels.extend(y.numpy().tolist())
    return np.concatenate(embeds), np.array(labels)


def plot_figure3(sam_model, sample_idx, dataset, device, class_names, pca_models,
                 shared_loader, save_path=None):
    sam_model.eval()
    frames, true_label = dataset[sample_idx]
    frames_b = frames.unsqueeze(0).to(device)
    with torch.no_grad():
        out, extra = sam_model(frames_b, return_attention=True, return_routing=True)

    fig = plt.figure(figsize=(20, 5))
    ax0 = fig.add_subplot(1, 4, 1)
    ax1 = fig.add_subplot(1, 4, 2)
    ax2 = fig.add_subplot(1, 4, 3)
    ax3 = fig.add_subplot(1, 4, 4, projection='polar')

    # (a) Attention overlay -- middle frame
    T = frames.shape[0]
    mid = T // 2
    frame_img = frames[mid].permute(1, 2, 0).numpy()
    frame_img = np.clip(frame_img * np.array([0.229, 0.224, 0.225]) +
                        np.array([0.485, 0.456, 0.406]), 0, 1)
    if 'spatial' in extra:
        am = extra['spatial'][0, mid, 0].cpu().numpy()
        am = cv2.resize(am, (frame_img.shape[1], frame_img.shape[0]), interpolation=cv2.INTER_CUBIC)
        am = (am - am.min()) / (am.max() - am.min() + 1e-8)
        overlay = frame_img * 0.4 + plt.cm.jet(am)[:, :, :3] * 0.6
    else:
        overlay = frame_img
    ax0.imshow(np.clip(overlay, 0, 1)); ax0.axis('off')
    ax0.set_title("(a) Attention localisation")

    # (b) PrimaryCaps formation -- mean capsule magnitude per spatial location
    primary = extra['primary'][0]                        # (N_primary, 8)
    Hp, Wp  = sam_model.FEAT_H, sam_model.FEAT_W
    n_types = sam_model.PRIM_CAPS
    mags_grid = primary.norm(dim=-1).cpu().numpy().reshape(Hp, Wp, n_types).mean(axis=-1)
    im1 = ax1.imshow(mags_grid, cmap='viridis')
    ax1.set_title("(b) PrimaryCaps formation\n({} types x {}x{} = {} caps)".format(
        n_types, Hp, Wp, n_types * Hp * Wp))
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, fraction=0.046)

    # (c) PCA capsule-space separability + silhouette scores
    sil_scores = {}
    for name, m in pca_models.items():
        emb, lab = get_capsule_embeddings(m, shared_loader, device)
        sil_scores[name] = (silhouette_score(emb, lab) if len(set(lab.tolist())) > 1
                            else float('nan'))
        if name == 'SAM-CapsNet':
            proj = PCA(n_components=2).fit_transform(emb)
            for ci in sorted(set(lab.tolist())):
                msk = lab == ci
                ax2.scatter(proj[msk, 0], proj[msk, 1], s=14, label=class_names[ci], alpha=0.7)
    title_lines = " | ".join("{}={:.3f}".format(k, v) for k, v in sil_scores.items())
    ax2.set_title("(c) Activity-capsule separability (PCA)\nsilhouette: {}".format(title_lines), fontsize=8)
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")

    # (d) Polar view for this sample
    v = out[0].cpu().numpy()                    # (num_classes, 16)
    mags_v = np.linalg.norm(v, axis=1)
    proj_v = PCA(n_components=2).fit_transform(v)
    angles = np.arctan2(proj_v[:, 1], proj_v[:, 0])
    pred = int(mags_v.argmax())
    colors_d = ['#2ca02c' if i == true_label else '#1f77b4' for i in range(len(mags_v))]
    ax3.scatter(angles, mags_v, c=colors_d, s=80, zorder=3)
    for i, cn in enumerate(class_names):
        ax3.annotate(cn, (angles[i], mags_v[i]), fontsize=6)
    ax3.set_title("(d) Polar view\npred={} (conf={:.3f})".format(class_names[pred], mags_v[pred]), fontsize=9)

    plt.suptitle("FIGURE 3 (reproduced) -- true class: {}".format(class_names[true_label]), fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    return sil_scores


if all(k in trained_models for k in ('full_sam', 'no_sam')) and 'baseline_cnn' in baseline_models:
    _, _, test_idx_f3 = make_stratified_splits(dataset, seed=42)
    sample_idx_f3  = test_idx_f3[0]
    f3_loader = DataLoader(Subset(dataset, test_idx_f3), batch_size=4, shuffle=False, num_workers=1)
    pca_models = {'SAM-CapsNet': trained_models['full_sam'],
                 'Pure CapsNet': trained_models['no_sam'],
                 'CNN baseline': baseline_models['baseline_cnn']}
    sil_scores = plot_figure3(trained_models['full_sam'], sample_idx_f3, dataset, DEVICE,
                              CLASS_NAMES, pca_models, f3_loader,
                              save_path=SAVE_DIR + '/figure3_interpretability.png')
    print("Silhouette scores:", sil_scores)
    print("Paper reports: SAM-CapsNet 0.641, Pure CapsNet 0.487, CNN baseline 0.219.")
else:
    print("Run Cells 10 and 10B first so trained_models/baseline_models are populated.")


## Cell 17 — Inter-Class Capsule Vector Analysis

(Basketball, Biking, Diving, GolfSwing, HorseRiding, SoccerJuggling; 938 clips total in the paper) with per-class angular mean/std and magnitude mean/std.

In [ ]:
FIG4_CLASSES = ['Basketball', 'Biking', 'Diving', 'GolfSwing', 'HorseRiding', 'SoccerJuggling']

def circular_mean_std_deg(angles_rad):
    """Circular mean/std in degrees via the mean resultant vector length R
    (standard circular-statistics formula; handles wrap-around / bimodality
    better than a naive linear std)."""
    cos_m, sin_m = np.cos(angles_rad).mean(), np.sin(angles_rad).mean()
    R = np.hypot(cos_m, sin_m)
    mean_angle = np.degrees(np.arctan2(sin_m, cos_m))
    std_angle  = np.degrees(np.sqrt(max(-2.0 * np.log(max(R, 1e-8)), 0.0)))
    return mean_angle, std_angle


def collect_class_capsule_vectors(model, dataset, device, class_names_wanted):
    """Runs every clip belonging to the requested classes (train+val+test
    pooled, matching the paper's 'full subset' framing for Fig. 4) through
    the model and keeps each clip's TRUE-class activity-capsule vector."""
    model.eval()
    wanted_idx = {ci for ci, cn in dataset.idx_to_class.items()
                  if cn in class_names_wanted or cn.replace('WithDog', '') in class_names_wanted}
    idxs = [i for i, (_, lbl) in enumerate(dataset.samples) if lbl in wanted_idx]
    loader = DataLoader(Subset(dataset, idxs), batch_size=4, shuffle=False, num_workers=1)
    vecs_by_class = {ci: [] for ci in wanted_idx}
    with torch.no_grad():
        for frames, labels in loader:
            frames = frames.to(device)
            out = model(frames)                       # (B, num_classes, 16)
            for i in range(frames.size(0)):
                ci = int(labels[i].item())
                vecs_by_class[ci].append(out[i, ci].cpu().numpy())
    return vecs_by_class


if 'full_sam' in trained_models:
    vecs_by_class = collect_class_capsule_vectors(trained_models['full_sam'], dataset, DEVICE, FIG4_CLASSES)
    all_vecs = np.concatenate([np.stack(v) for v in vecs_by_class.values() if len(v) > 0])
    pca4 = PCA(n_components=2).fit(all_vecs)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': 'polar'})
    summary_rows = []
    for ax, ci in zip(axes.flat, sorted(vecs_by_class.keys())):
        vecs = np.stack(vecs_by_class[ci])
        n    = len(vecs)
        proj = pca4.transform(vecs)
        angles = np.arctan2(proj[:, 1], proj[:, 0])
        mags   = np.linalg.norm(vecs, axis=1)
        mean_ang, std_ang = circular_mean_std_deg(angles)
        ax.scatter(angles, mags, s=12, alpha=0.6)
        cname = dataset.idx_to_class[ci]
        ax.set_title("{} (n={})\nangle {:.1f}+/-{:.1f} deg, mag {:.3f}+/-{:.3f}".format(
            cname, n, mean_ang, std_ang, mags.mean(), mags.std()), fontsize=9)
        summary_rows.append(dict(class_name=cname, n=n, mean_angle_deg=float(mean_ang),
                                 angle_std_deg=float(std_ang), mean_magnitude=float(mags.mean()),
                                 magnitude_std=float(mags.std())))

    plt.suptitle("FIGURE 4 (reproduced) -- per-class activity-capsule distributions (total n={})".format(
        sum(r['n'] for r in summary_rows)), fontweight='bold')
    plt.tight_layout()
    plt.savefig(SAVE_DIR + '/figure4_interclass_capsule_analysis.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    print("\n{:<16} {:>6} {:>16} {:>16}".format("Class", "n", "Angle (deg)", "Magnitude"))
    for r in summary_rows:
        print("{:<16} {:>6} {:>8.1f}+/-{:<5.1f} {:>7.3f}+/-{:<5.3f}".format(
            r['class_name'], r['n'], r['mean_angle_deg'], r['angle_std_deg'],
            r['mean_magnitude'], r['magnitude_std']))

    with open(SAVE_DIR + '/figure4_summary.json', 'w') as f:
        json.dump(summary_rows, f, indent=2)
    print("\nPaper (Sec. III-D): Diving tightest (41.9+/-3.9 deg, mag 0.843+/-0.072);")
    print("HorseRiding widest/bimodal (41.5 deg spread, two clusters).")
else:
    print("Run Cell 10 (full ablation) first so trained_models['full_sam'] exists.")


## Cell 18 — Confusion-Matrix Comparison: CNN+SAM Baseline vs SAM-CapsNet (Fig. 5)

Uses the `baseline_last_preds`/`baseline_last_gt` and `last_run_preds`/`last_run_gt` stashes added to Cells 10 and 10B above (last-seed, single-run predictions).

In [ ]:
def plot_confusion_matrix_pair(gt_a, preds_a, gt_b, preds_b, class_names,
                               title_a, title_b, save_path=None):
    cm_a = confusion_matrix(gt_a, preds_a)
    cm_b = confusion_matrix(gt_b, preds_b)
    acc_a = (np.array(preds_a) == np.array(gt_a)).mean() * 100
    acc_b = (np.array(preds_b) == np.array(gt_b)).mean() * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2 * len(class_names), len(class_names) - 1))
    for ax, cm, ttl, acc in [(ax1, cm_a, title_a, acc_a), (ax2, cm_b, title_b, acc_b)]:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                   xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
        ax.set_title("{}\nAcc = {:.2f}%".format(ttl, acc))
        ax.set_ylabel("True"); ax.set_xlabel("Predicted")
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show(); plt.close()
    return acc_a, acc_b


if 'cnn_sam' in baseline_last_preds and 'full_sam' in last_run_preds:
    acc_baseline, acc_proposed = plot_confusion_matrix_pair(
        baseline_last_gt['cnn_sam'], baseline_last_preds['cnn_sam'],
        last_run_gt['full_sam'],     last_run_preds['full_sam'],
        CLASS_NAMES,
        "(a) Baseline CNN + Multi-Scale SAM", "(b) SAM-CapsNet (Proposed)",
        save_path=SAVE_DIR + '/figure5_confusion_comparison.png')
    print("This run: baseline {:.2f}% vs SAM-CapsNet {:.2f}%".format(acc_baseline, acc_proposed))
    print("Paper reports 77.73% vs 86.97% on its reference run (Sec. III-E).")
else:
    print("Run Cells 10 and 10B first so last-seed preds/gt for 'full_sam' and 'cnn_sam' exist.")
